In [291]:
# =============================================================================
# EMBER NOTEBOOK ANALYZER - STEP-BY-STEP PRESENTATION
# =============================================================================
# This notebook demonstrates the complete pipeline for analyzing Jupyter notebooks
# and generating educational walkthroughs with component linking

import os
import getpass
from dotenv import load_dotenv

load_dotenv()
os.environ["TAVILY_API_KEY"] 
os.environ["OPENAI_API_KEY"] 
os.environ["LANGCHAIN_TRACING_V2"] 
os.environ["LANGCHAIN_PROJECT"] 
os.environ["LANGCHAIN_API_KEY"] 

from openai import OpenAI
from langsmith.wrappers import wrap_openai
from langsmith import traceable

openai_client = wrap_openai(OpenAI())

print("🚀 Environment configured successfully!")
print("📝 Ready for step-by-step notebook analysis")
print("=" * 60)

🚀 Environment configured successfully!
📝 Ready for step-by-step notebook analysis


In [292]:
# =============================================================================
# EMBER OUTPUT MANAGEMENT SYSTEM
# =============================================================================
# Comprehensive system for saving all analysis outputs to ember_output directory

import os
import json
import shutil
from datetime import datetime
from pathlib import Path
from typing import Dict, Any, Optional

class EmberOutputManager:
    """Manages all output files for Ember analysis runs"""
    
    def __init__(self, base_dir: str = "ember_output"):
        self.base_dir = Path(base_dir)
        self.current_run_dir = None
        self.run_timestamp = None
        
    def start_new_run(self, notebook_name: str) -> str:
        """Start a new analysis run with timestamped directory"""
        self.run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        notebook_base = Path(notebook_name).stem
        self.current_run_dir = self.base_dir / f"{notebook_base}_{self.run_timestamp}"
        
        # Create directory structure
        self.current_run_dir.mkdir(parents=True, exist_ok=True)
        (self.current_run_dir / "step1_extraction").mkdir(exist_ok=True)
        (self.current_run_dir / "step2_analysis").mkdir(exist_ok=True)
        (self.current_run_dir / "step3_components").mkdir(exist_ok=True)
        (self.current_run_dir / "step4_deep_descriptions").mkdir(exist_ok=True)
        (self.current_run_dir / "step5_walkthrough").mkdir(exist_ok=True)
        (self.current_run_dir / "debug").mkdir(exist_ok=True)
        
        # Create run info file
        run_info = {
            "notebook": notebook_name,
            "timestamp": self.run_timestamp,
            "start_time": datetime.now().isoformat(),
            "steps_completed": []
        }
        
        with open(self.current_run_dir / "run_info.json", 'w') as f:
            json.dump(run_info, f, indent=2)
        
        print(f"📁 Started new run: {self.current_run_dir}")
        return str(self.current_run_dir)
    
    def _make_json_serializable(self, obj):
        """Convert objects to JSON serializable format"""
        if hasattr(obj, 'model_dump'):
            # Handle Pydantic models
            data = obj.model_dump()
            return self._convert_datetime_to_string(data)
        elif isinstance(obj, datetime):
            return obj.isoformat()
        elif isinstance(obj, dict):
            return self._convert_datetime_to_string(obj)
        elif isinstance(obj, list):
            return [self._make_json_serializable(item) for item in obj]
        else:
            return obj
    
    def _convert_datetime_to_string(self, data):
        """Recursively convert datetime objects to ISO strings"""
        if isinstance(data, dict):
            result = {}
            for key, value in data.items():
                if isinstance(value, datetime):
                    result[key] = value.isoformat()
                elif isinstance(value, dict):
                    result[key] = self._convert_datetime_to_string(value)
                elif isinstance(value, list):
                    result[key] = [self._convert_datetime_to_string(item) if isinstance(item, (dict, datetime)) else item for item in value]
                else:
                    result[key] = value
            return result
        elif isinstance(data, datetime):
            return data.isoformat()
        else:
            return data
    
    def save_step_output(self, step: str, filename: str, content: Any, description: str = ""):
        """Save output for a specific step"""
        if not self.current_run_dir:
            raise ValueError("No active run. Call start_new_run() first.")
        
        step_dir = self.current_run_dir / f"step{step}"
        step_dir.mkdir(exist_ok=True)
        
        filepath = step_dir / filename
        
        # Handle different content types
        if isinstance(content, (dict, list)):
            with open(filepath, 'w', encoding='utf-8') as f:
                json.dump(content, f, indent=2)
        elif isinstance(content, str):
            with open(filepath, 'w', encoding='utf-8') as f:
                f.write(content)
        else:
            # For objects with model_dump method or complex objects
            try:
                serializable_content = self._make_json_serializable(content)
                with open(filepath, 'w', encoding='utf-8') as f:
                    json.dump(serializable_content, f, indent=2)
            except Exception as e:
                # Fallback to string representation
                print(f"⚠️  Warning: Could not serialize {filename} as JSON, saving as string: {e}")
                with open(filepath, 'w', encoding='utf-8') as f:
                    f.write(str(content))
        
        # Update run info
        self._update_run_info(step, filename, description)
        
        print(f"💾 Saved {filename} to step{step}/ - {description}")
        return str(filepath)
    
    def save_debug_info(self, filename: str, content: Any, description: str = ""):
        """Save debugging information"""
        if not self.current_run_dir:
            return
        
        debug_dir = self.current_run_dir / "debug"
        filepath = debug_dir / filename
        
        try:
            if isinstance(content, (dict, list)):
                with open(filepath, 'w', encoding='utf-8') as f:
                    json.dump(content, f, indent=2)
            elif isinstance(content, str):
                with open(filepath, 'w', encoding='utf-8') as f:
                    f.write(content)
            else:
                # Try to serialize complex objects
                serializable_content = self._make_json_serializable(content)
                with open(filepath, 'w', encoding='utf-8') as f:
                    json.dump(serializable_content, f, indent=2)
        except Exception as e:
            # Fallback to string representation
            with open(filepath, 'w', encoding='utf-8') as f:
                f.write(str(content))
        
        print(f"🐛 Debug info: {filename} - {description}")
    
    def _update_run_info(self, step: str, filename: str, description: str):
        """Update run information"""
        run_info_path = self.current_run_dir / "run_info.json"
        
        with open(run_info_path, 'r') as f:
            run_info = json.load(f)
        
        step_info = {
            "step": step,
            "filename": filename,
            "description": description,
            "timestamp": datetime.now().isoformat()
        }
        
        run_info["steps_completed"].append(step_info)
        run_info["last_updated"] = datetime.now().isoformat()
        
        with open(run_info_path, 'w') as f:
            json.dump(run_info, f, indent=2)
    
    def create_run_summary(self):
        """Create a comprehensive summary of the run"""
        if not self.current_run_dir:
            return
        
        summary = {
            "run_directory": str(self.current_run_dir),
            "timestamp": self.run_timestamp,
            "files_created": [],
            "step_summaries": {}
        }
        
        # Collect all files
        for step_dir in self.current_run_dir.iterdir():
            if step_dir.is_dir() and step_dir.name.startswith('step'):
                step_files = []
                for file_path in step_dir.glob('*'):
                    if file_path.is_file():
                        step_files.append({
                            "filename": file_path.name,
                            "size": file_path.stat().st_size,
                            "modified": datetime.fromtimestamp(file_path.stat().st_mtime).isoformat()
                        })
                summary["step_summaries"][step_dir.name] = step_files
                summary["files_created"].extend([str(f) for f in step_dir.glob('*')])
        
        # Save summary
        with open(self.current_run_dir / "run_summary.json", 'w') as f:
            json.dump(summary, f, indent=2)
        
        print(f"\n📊 RUN SUMMARY")
        print(f"Directory: {self.current_run_dir}")
        print(f"Total files: {len(summary['files_created'])}")
        for step, files in summary["step_summaries"].items():
            print(f"  {step}: {len(files)} files")
    
    def clean_old_runs(self, keep_latest: int = 3):
        """Clean up old run directories, keeping only the latest N runs"""
        if not self.base_dir.exists():
            return
        
        # Get all run directories
        run_dirs = [d for d in self.base_dir.iterdir() if d.is_dir() and '_' in d.name]
        run_dirs.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # Remove old runs
        for old_dir in run_dirs[keep_latest:]:
            print(f"🗑️ Cleaning old run: {old_dir.name}")
            shutil.rmtree(old_dir)
    
    def list_runs(self):
        """List all available runs"""
        if not self.base_dir.exists():
            print("No runs found")
            return []
        
        runs = []
        for run_dir in self.base_dir.iterdir():
            if run_dir.is_dir():
                run_info_path = run_dir / "run_info.json"
                if run_info_path.exists():
                    with open(run_info_path) as f:
                        info = json.load(f)
                    runs.append({
                        "directory": run_dir.name,
                        "notebook": info.get("notebook", "unknown"),
                        "timestamp": info.get("timestamp", "unknown"),
                        "steps": len(info.get("steps_completed", []))
                    })
        
        runs.sort(key=lambda x: x["timestamp"], reverse=True)
        
        print(f"\n📋 AVAILABLE RUNS ({len(runs)}):")
        for run in runs:
            print(f"  {run['directory']} - {run['notebook']} ({run['steps']} steps)")
        
        return runs

# Global output manager instance
output_manager = EmberOutputManager()

def start_analysis_run(notebook_name: str):
    """Convenience function to start a new analysis run"""
    return output_manager.start_new_run(notebook_name)

def save_step(step: str, filename: str, content: Any, description: str = ""):
    """Convenience function to save step output"""
    return output_manager.save_step_output(step, filename, content, description)

def save_debug(filename: str, content: Any, description: str = ""):
    """Convenience function to save debug info"""
    return output_manager.save_debug_info(filename, content, description)

def finish_run():
    """Convenience function to finish a run"""
    output_manager.create_run_summary()
    output_manager.clean_old_runs(keep_latest=5)  # Keep 5 most recent runs

print("🎯 Ember Output Management System initialized!")
print("📁 Use start_analysis_run('notebook.ipynb') to begin")
print("💾 Use save_step(step, filename, content, description) to save outputs")
print("🏁 Use finish_run() when analysis is complete")

🎯 Ember Output Management System initialized!
📁 Use start_analysis_run('notebook.ipynb') to begin
💾 Use save_step(step, filename, content, description) to save outputs
🏁 Use finish_run() when analysis is complete


# 📁 Configuration: Set Your Notebook to Analyze

**Change the filename below to analyze any Jupyter notebook.**  
This variable will be used throughout all subsequent steps.

# 🎯 NOTEBOOK TO ANALYZE - CHANGE THIS!
NOTEBOOK_TO_ANALYZE = 'tiny_demo.ipynb'

# Global variables to store results from each step
extracted_code = None
extracted_markdown = None
explanation = None
final_state = None
improved_state = None
walkthrough = None

print(f"📖 Target notebook: {NOTEBOOK_TO_ANALYZE}")
print("🔄 All results will build on this notebook")
print("💾 To start fresh: restart kernel and run all cells")
print("\n" + "="*50)
print("FILES TO DELETE FOR FRESH RUN:")
print(f"- {NOTEBOOK_TO_ANALYZE.replace('.ipynb', '.py')}")
print(f"- {NOTEBOOK_TO_ANALYZE.replace('.ipynb', '.md')}")
print(f"- analysis_{NOTEBOOK_TO_ANALYZE.replace('.ipynb', '')}/ (entire folder)")
print("="*50)

# 🔧 STEP 1: Python Code and Markdown Extraction

This step separates the Jupyter notebook into:
- **Python code**: All executable code cells combined
- **Markdown content**: All documentation and text cells

This forms the foundation for all subsequent analysis steps.

In [293]:
import json
import sys

def separate_ipynb(input_file, output_py=None, output_md=None):
    """
    Separate an ipynb file into .py and .md files while preserving order.
    
    Args:
        input_file: Path to the input .ipynb file
        output_py: Path for the output .py file (defaults to input_file.py)
        output_md: Path for the output .md file (defaults to input_file.md)
    """
    # Set default output filenames if not provided
    if output_py is None:
        output_py = input_file.replace('.ipynb', '.py')
    if output_md is None:
        output_md = input_file.replace('.ipynb', '.md')
    
    # Read the notebook
    with open(input_file, 'r', encoding='utf-8') as f:
        notebook = json.load(f)
    
    py_content = []
    md_content = []
    
    # Add header comments
    py_content.append(f"# Generated from {input_file}")
    py_content.append("# Python cells extracted in order\n")
    
    md_content.append(f"# Generated from {input_file}")
    md_content.append("## Markdown cells extracted in order\n")
    
    # Process cells in order
    for i, cell in enumerate(notebook.get('cells', [])):
        cell_type = cell.get('cell_type', '')
        source = cell.get('source', [])
        
        # Join source lines if it's a list
        if isinstance(source, list):
            source_text = ''.join(source)
        else:
            source_text = source
        
        if cell_type == 'code':
            # Add cell separator comment
            py_content.append(f"\n# ========== Cell {i+1} ==========")
            py_content.append(source_text.rstrip())
            
        elif cell_type == 'markdown':
            # Add cell separator
            md_content.append(f"\n---\n<!-- Cell {i+1} -->\n")
            md_content.append(source_text.rstrip())
    
    # Write the .py file
    with open(output_py, 'w', encoding='utf-8') as f:
        f.write('\n'.join(py_content))
    
    # Write the .md file
    with open(output_md, 'w', encoding='utf-8') as f:
        f.write('\n'.join(md_content))
        
    extracted_code = '\n'.join(py_content)
    extracted_markdown = '\n'.join(md_content)
    
    print(f"Successfully separated '{input_file}' into:")
    print(f"  - Python file: {output_py}")
    print(f"  - Markdown file: {output_md}")
    
    return extracted_code, extracted_markdown

# Example usage:
# separate_ipynb('example.ipynb')
# or with custom output names:
# separate_ipynb('example.ipynb', 'my_code.py', 'my_docs.md')

# 🧠 STEP 2: Initial Analysis and Block Structure

This step analyzes the extracted Python code to:
- Create a **high-level overview** of the code's purpose
- Identify logical **code blocks** (imports, classes, functions, etc.)
- Establish **dependencies** between blocks
- Determine **execution order** for understanding

The analysis uses LLM-powered structured output to ensure consistent results.

  1. JSON output format - structured and parseable
  2. Clearer dependency tracking - both dependencies and dependents listed per block
  3. Edge types - distinguishes "uses" vs "bidirectional" relationships
  4. Execution order - helps understand the flow
  5. More specific guidelines - for consistent block identification

In [294]:
INITIAL_EXPLANATION_PROMPT = """You are an expert Python tutor analyzing the attached Python code to create a comprehensive explanation graph.

Your task is to decompose the code into logical blocks that form a dependency graph, explaining how each block functions and relates to others.

## Guidelines:
1. Each block should represent a cohesive unit of functionality
2. Blocks can contain: imports, class definitions, function definitions, global variables, or main execution code
3. Edge directions indicate dependencies: A -> B means "A depends on/uses B"
4. Bidirectional edges (A <-> B) indicate mutual dependencies
5. Every line of code must belong to exactly one block
6. Preserve the complete code in each block - do not split nested structures

## Target audience:
Someone with basic Python knowledge who needs to understand:
- What each section of code does
- How different parts interact
- The overall data and control flow

## Required JSON output structure:
{{
  "overview": "A 2-3 sentence summary of the code's overall purpose and functionality",
  "blocks": [
    {{
      "id": "1",
      "name": "Descriptive Block Name",
      "description": "Detailed explanation of what this block does, its role in the application, and key components within it. Include how it processes data or provides functionality.",
      "content": "# The actual Python code for this block\\n# Including all related lines",
      "dependencies": ["2", "3"],
      "dependents": ["4"]
    }}
  ],
  "edges": [
    {{"from": "1", "to": "2", "type": "uses"}},
    {{"from": "3", "to": "1", "type": "uses"}},
    {{"from": "4", "to": "5", "type": "bidirectional"}}
  ],
  "execution_order": ["1", "2", "3", "4", "5"]
}}

## Block identification hints:
- Group imports together if they serve similar purposes
- Keep class definitions with their methods
- Group related utility functions
- Separate initialization/configuration from main logic
- Consider data flow when deciding block boundaries

Python Code to analyze:

{extracted_code}
"""

In [295]:
NOTEBOOK_TO_ANALYZE = "multi_agent.ipynb"

# 🚀 START NEW ANALYSIS RUN
print("🎯 STARTING EMBER ANALYSIS")
print("="*60)
start_analysis_run(NOTEBOOK_TO_ANALYZE)

# 🔨 STEP 1: RUN EXTRACTION
print("\n🔍 STEP 1: EXTRACTING CODE AND MARKDOWN")
print("="*50)

global extracted_code, extracted_markdown

# Extract Python and Markdown from the target notebook
extracted_code, extracted_markdown = separate_ipynb(NOTEBOOK_TO_ANALYZE)

print(f"✅ Python code: {len(extracted_code.split())} lines")
print(f"✅ Markdown: {len(extracted_markdown.split())} lines")
print(f"✅ Files saved: {NOTEBOOK_TO_ANALYZE.replace('.ipynb', '.py')}, {NOTEBOOK_TO_ANALYZE.replace('.ipynb', '.md')}")

# Save extraction outputs
save_step("1_extraction", "extracted_code.py", extracted_code, "Python code extracted from notebook")
save_step("1_extraction", "extracted_markdown.md", extracted_markdown, "Markdown content extracted from notebook")

# Create extraction summary
extraction_summary = {
    "notebook": NOTEBOOK_TO_ANALYZE,
    "python_code_length": len(extracted_code),
    "python_word_count": len(extracted_code.split()),
    "markdown_length": len(extracted_markdown),
    "markdown_word_count": len(extracted_markdown.split()),
    "files_created": [
        NOTEBOOK_TO_ANALYZE.replace('.ipynb', '.py'),
        NOTEBOOK_TO_ANALYZE.replace('.ipynb', '.md')
    ]
}
save_step("1_extraction", "extraction_summary.json", extraction_summary, "Summary of extraction process")

# Show preview of extracted code
print(f"\n📄 PYTHON CODE PREVIEW (first 500 chars):")
print("─" * 50)
preview = extracted_code[:500] + "..." if len(extracted_code) > 500 else extracted_code
print(preview)

# Save preview
save_step("1_extraction", "code_preview.txt", preview, "Preview of extracted Python code")

🎯 STARTING EMBER ANALYSIS
📁 Started new run: ember_output/multi_agent_20250806_072751

🔍 STEP 1: EXTRACTING CODE AND MARKDOWN
Successfully separated 'multi_agent.ipynb' into:
  - Python file: multi_agent.py
  - Markdown file: multi_agent.md
✅ Python code: 2434 lines
✅ Markdown: 2676 lines
✅ Files saved: multi_agent.py, multi_agent.md
💾 Saved extracted_code.py to step1_extraction/ - Python code extracted from notebook
💾 Saved extracted_markdown.md to step1_extraction/ - Markdown content extracted from notebook
💾 Saved extraction_summary.json to step1_extraction/ - Summary of extraction process

📄 PYTHON CODE PREVIEW (first 500 chars):
──────────────────────────────────────────────────
# Generated from multi_agent.ipynb
# Python cells extracted in order


# ========== Cell 5 ==========
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")
os.environ["TAVILY_API_KEY"] = getpass.getpass("TAVILY_API_KEY")

# ========== Cell 9 ==========
from langchain_c

'ember_output/multi_agent_20250806_072751/step1_extraction/code_preview.txt'

  1. Sets up LangChain with OpenAI - using ChatOpenAI with gpt-4o-mini
  2. Defines Pydantic models - matching the JSON structure from your prompt
  3. Uses structured output - with_structured_output() ensures consistent JSON responses
  4. Adds LangSmith tracing - @traceable decorator tracks the function execution
  5. Creates analyze_code function - combines extraction and analysis in one step

  The structured output approach ensures you get properly formatted JSON responses that match your schema exactly. The
  Edge model uses alias="from" since "from" is a Python keyword.

In [296]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
import json

# Initialize the OpenAI model with structured output
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Define the structure for the response
from pydantic import BaseModel, Field
from typing import List

class CodeBlock(BaseModel):
    id: str = Field(description="Unique identifier for the block")
    name: str = Field(description="Descriptive name for the block")
    description: str = Field(description="Detailed explanation of what this block does")
    content: str = Field(description="The actual Python code for this block")
    dependencies: List[str] = Field(default_factory=list, description="IDs of blocks this block depends on")
    dependents: List[str] = Field(default_factory=list, description="IDs of blocks that depend on this block")

class Edge(BaseModel):
    from_id: str = Field(alias="from", description="Source block ID")
    to: str = Field(description="Target block ID")
    type: str = Field(description="Type of relationship: 'uses' or 'bidirectional'")

class CodeExplanation(BaseModel):
    overview: str = Field(description="2-3 sentence summary of the code's purpose")
    blocks: List[CodeBlock] = Field(description="List of code blocks")
    edges: List[Edge] = Field(description="Dependency edges between blocks")
    execution_order: List[str] = Field(description="Suggested order for understanding the code")

# Create structured LLM
structured_llm = llm.with_structured_output(CodeExplanation)

# Function to analyze code
@traceable  # This will trace the function in LangSmith
def analyze_code(ipynb_file: str) -> CodeExplanation:
    """Analyze a Jupyter notebook and return structured explanation"""
    
    # Extract code from notebook
    python_code, _ = separate_ipynb(ipynb_file)
    
    # Format the prompt
    formatted_prompt = INITIAL_EXPLANATION_PROMPT.format(extracted_code=python_code)
    
    # Create messages
    messages = [
        SystemMessage(content=formatted_prompt),
        HumanMessage(content="Analyze this code and provide the structured explanation.")
    ]
    
    # Get structured response
    result = structured_llm.invoke(messages)
    
    return result

# Example usage
# explanation = analyze_code('tiny_demo.ipynb')
# print(json.dumps(explanation.model_dump(), indent=2))

In [297]:
# 🔨 STEP 2: RUN INITIAL ANALYSIS
print("\n🧠 STEP 2: ANALYZING BLOCKS AND STRUCTURE")
print("="*50)

global explanation

# Run initial analysis on the extracted code
explanation = analyze_code(NOTEBOOK_TO_ANALYZE)

print(f"📊 OVERVIEW:")
print(f"   {explanation.overview}")
print(f"\n📦 FOUND {len(explanation.blocks)} BLOCKS:")
for i, block in enumerate(explanation.blocks, 1):
    print(f"   {i}. {block.name} (ID: {block.id})")
    print(f"      Dependencies: {block.dependencies if block.dependencies else 'None'}")

print(f"\n🔗 DEPENDENCY EDGES: {len(explanation.edges)}")
for edge in explanation.edges:
    print(f"   Block {edge.from_id} → Block {edge.to} ({edge.type})")

print(f"\n📋 EXECUTION ORDER: {' → '.join(explanation.execution_order)}")

# Save complete analysis
save_step("2_analysis", "code_explanation.json", explanation, "Complete block-level analysis")

# Save individual components
save_step("2_analysis", "overview.txt", explanation.overview, "High-level code overview")

# Create blocks summary
blocks_summary = []
for block in explanation.blocks:
    blocks_summary.append({
        "id": block.id,
        "name": block.name,
        "description": block.description,
        "dependencies": block.dependencies,
        "dependents": block.dependents,
        "code_length": len(block.content),
        "code_lines": len(block.content.split('\n'))
    })
save_step("2_analysis", "blocks_summary.json", blocks_summary, "Summary of all code blocks")

# Save dependency graph
dependency_data = {
    "edges": [{"from": edge.from_id, "to": edge.to, "type": edge.type} for edge in explanation.edges],
    "execution_order": explanation.execution_order,
    "total_blocks": len(explanation.blocks),
    "total_edges": len(explanation.edges)
}
save_step("2_analysis", "dependency_graph.json", dependency_data, "Block dependency relationships")

# Show first block details
if explanation.blocks:
    first_block = explanation.blocks[0]
    print(f"\n📄 FIRST BLOCK PREVIEW ({first_block.name}):")
    print("─" * 50)
    first_block_preview = first_block.content[:300] + "..." if len(first_block.content) > 300 else first_block.content
    print(first_block_preview)
    
    # Save first block details
    save_step("2_analysis", "first_block_preview.txt", first_block_preview, "Preview of first code block")
    save_step("2_analysis", "first_block_full.py", first_block.content, "Complete first block code")


🧠 STEP 2: ANALYZING BLOCKS AND STRUCTURE
Successfully separated 'multi_agent.ipynb' into:
  - Python file: multi_agent.py
  - Markdown file: multi_agent.md


📊 OVERVIEW:
   This code implements a Retrieval-Augmented Generation (RAG) system that utilizes various language models and document loaders to process and respond to queries about student loans. It integrates multiple agents for research, response generation, and document management, allowing for a collaborative approach to generating informative responses based on user queries.

📦 FOUND 24 BLOCKS:
   1. Environment Setup (ID: 1)
      Dependencies: None
   2. Document Loading (ID: 2)
      Dependencies: ['1']
   3. Text Processing (ID: 3)
      Dependencies: ['2']
   4. Chunk Length Calculation (ID: 4)
      Dependencies: ['3']
   5. Embedding Model Initialization (ID: 5)
      Dependencies: ['3']
   6. Vector Store Creation (ID: 6)
      Dependencies: ['5']
   7. Retriever Initialization (ID: 7)
      Dependencies: ['6']
   8. Prompt Template Definition (ID: 8)
      Dependencies: ['7']
   9. Chat Model Initialization (ID: 9)
      Dependencies: ['8']
   10. State Definition and RAG

# 🔍 STEP 3: Component Extraction and Dependency Resolution

This step performs detailed analysis of each block to:
- **Extract individual components** (imports, classes, methods, functions, variables, expressions)
- **Establish relationships** between components (parent/child, calls/called-by)
- **Resolve cross-block dependencies** to enhance component descriptions
- **Build comprehensive registries** for efficient component lookup

The LangGraph workflow manages state transitions and ensures proper dependency resolution.

1. Enhanced Prompt (COMPONENT_EXTRACTION_PROMPT):
    - Structured to extract meaningful components from code blocks
    - Identifies relationships between components (parent/child, calls/called-by)
    - Preserves all code with proper attribution
  2. Token Optimization Strategy (format_connected_blocks):
    - Only includes summaries of connected blocks (not full code)
    - Shows key functions/classes from connected blocks
    - Limits description previews to 200 characters
    - Shows only first 5 key items per block
  3. Component Types Supported:
    - imports, classes, methods, functions, variables, expressions, decorators
    - Tracks dependencies within and across blocks
  4. Structured Output:
    - Uses Pydantic models for consistent JSON responses
    - Tracks 10 attributes per component for comprehensive analysis

  The prompt balances coverage with token efficiency by:
  - Only including connected block summaries (not full code)
  - Extracting key identifiers from connected blocks
  - Using concise descriptions
  - Limiting preview lengths

In [298]:
# Import required modules
from typing import Optional, List
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langsmith import traceable
import json

# Enhanced prompt for component extraction from code blocks
COMPONENT_EXTRACTION_PROMPT = """You are an expert Python code analyzer tasked with decomposing a code block into its meaningful components.

## Context:
{overview}

## Current Block:
**Description:** {block_description}
**Block ID:** {block_id}

## CRITICAL TASK:
Analyze the code block below and identify ALL meaningful components. Each line of code must be attributed to exactly one component.

## Component Types to Identify:

### 1. IMPORTS (Group strategically)
- **Single library**: `import pandas` 
- **Multi-module**: `from sklearn.model_selection import train_test_split, cross_val_score`
- **Local imports**: `from .utils import helper_function`
- **Aliased imports**: `import numpy as np`

### 2. CLASSES (Include all methods and attributes)
- **Class definition**: `class MyClass:`
- **Constructor**: `def __init__(self, param):`
- **Methods**: `def method_name(self):`
- **Properties**: `@property def value(self):`
- **Class variables**: Variables defined at class level

### 3. FUNCTIONS (Standalone functions outside classes)
- **Regular functions**: `def function_name():`
- **Async functions**: `async def async_function():`
- **Lambda functions**: `lambda x: x + 1`
- **Decorated functions**: Functions with decorators

### 4. VARIABLES (Assignment statements)
- **Simple assignment**: `variable = value`
- **Multiple assignment**: `a, b = values`
- **Constants**: `CONFIG = {{...}}`
- **Complex objects**: `model = Pipeline([...])`

### 5. EXPRESSIONS (Standalone executable statements)
- **Function calls**: `print("hello")`
- **Method calls**: `obj.method()`
- **List/dict operations**: `items.append(value)`
- **Calculations**: `result = a + b * c`

### 6. DECORATORS (When defined, not when applied)
- **Function decorators**: `@decorator`
- **Class decorators**: Decorating classes
- **Custom decorators**: Decorator definitions

## HIERARCHY AWARENESS:
- **Parent Components**: Classes contain methods; modules contain functions
- **Child Components**: Methods belong to classes; nested functions to outer functions
- **Scope Tracking**: Track which scope each component belongs to

## EXPRESSION GRANULARITY EXAMPLES:

### ✅ CORRECT Granularity:
```python
# These are SEPARATE components:
data = load_data()           # Component 1: Variable assignment
cleaned = clean_text(data)   # Component 2: Variable assignment  
model.fit(cleaned)          # Component 3: Method call expression
```

### ❌ INCORRECT Granularity:
```python
# DON'T group these as one component:
data = load_data()
cleaned = clean_text(data)
model.fit(cleaned)
# Each line is a separate component!
```

### Complex Assignment Examples:
```python
# Component: Variable with complex object
pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', SVC())
])

# Component: Multiple assignment
X_train, X_test, y_train, y_test = train_test_split(X, y)

# Component: Method chain assignment  
result = df.groupby('category').agg({{'value': 'mean'}}).reset_index()
```

## IMPORT SPECIFICITY RULES:
- Group related imports: `import os, sys, json` → Single component
- Separate unrelated: `import pandas` + `import torch` → Two components  
- Complex from-imports: `from sklearn.metrics import accuracy_score, precision_score` → One component
- Local vs external: Keep separate when functionality differs significantly

## DEDUPLICATION GUIDANCE:
- If same function appears multiple times, only create ONE component
- For method overloading, create separate components if significantly different
- For identical imports, merge into single component
- For similar variables (like config variants), keep separate if used differently

## For each component, provide:
1. **name**: Descriptive name (e.g., "load_data_function", "Pipeline_setup", "model_training_call")
2. **number**: Sequential number within this block
3. **type**: One of: import, class, method, function, variable, expression, decorator
4. **library**: Associated library (e.g., "pandas", "sklearn") or "built-in" or "local"
5. **code**: The exact code for this component (preserve formatting)
6. **description**: Brief explanation focusing on PURPOSE and ROLE
7. **parent**: Parent component number (methods → class, nested functions → outer function)
8. **children**: List of child component numbers  
9. **calls**: Component numbers this component calls/uses
10. **called_by**: Component numbers that call/use this component

## ADVANCED RELATIONSHIP EXAMPLES:

### Class Hierarchy:
```python
class DataProcessor:        # Component 1 (parent: None)
    def __init__(self):     # Component 2 (parent: 1)
        self.data = []      # Part of Component 2
    
    def process(self):      # Component 3 (parent: 1)
        return cleaned      # Part of Component 3
```

### Function Calls:
```python
def helper():              # Component 1
    return "processed"

def main():               # Component 2 (calls: [1])
    result = helper()     # This shows Component 2 calls Component 1
    return result
```

## Code to analyze:
```python
{block_code}
```

## Connected Blocks Context:
{connected_blocks_context}

## QUALITY CHECKLIST:
- [ ] Every line of code is attributed to exactly one component
- [ ] Parent-child relationships are correctly identified
- [ ] Import grouping follows specificity rules
- [ ] Expression granularity is appropriate
- [ ] Component names are descriptive and unique
- [ ] Library attributions are accurate
- [ ] Cross-component dependencies are identified

## Output Format:
Return a JSON structure with a "components" array containing the component objects as specified above."""

# Helper function to format connected blocks info concisely - using string concatenation to avoid f-string issues
def format_connected_blocks(edges, blocks, current_block_id):
    """Format connected blocks information concisely"""
    connected_info = []
    
    for edge in edges:
        # Handle Edge objects and dictionaries more safely
        try:
            if hasattr(edge, 'from_id'):
                # It's an Edge object - use the actual attribute names
                from_id = edge.from_id
                to_id = edge.to
            elif isinstance(edge, dict):
                # It's a dictionary - try different possible keys
                from_id = edge.get('from') or edge.get('from_id') or edge.get('source')
                to_id = edge.get('to') or edge.get('target')
            else:
                # Try to get it as a Pydantic model
                edge_dict = edge.model_dump() if hasattr(edge, 'model_dump') else vars(edge)
                from_id = edge_dict.get('from') or edge_dict.get('from_id') or edge_dict.get('source')
                to_id = edge_dict.get('to') or edge_dict.get('target')
        except Exception as e:
            print("Warning: Could not parse edge " + str(edge) + ": " + str(e))
            continue
        
        if not from_id or not to_id:
            continue
            
        # Find blocks connected to current block
        connected_id = None
        relation = None
        
        if from_id == current_block_id:
            connected_id = to_id
            relation = "uses"
        elif to_id == current_block_id:
            connected_id = from_id
            relation = "used by"
            
        if connected_id and relation:
            # Find the connected block
            for block in blocks:
                try:
                    block_id = block.get('id') if isinstance(block, dict) else (block.id if hasattr(block, 'id') else str(block))
                    if block_id == connected_id:
                        # Create concise summary safely using string concatenation
                        block_name = block.get('name') if isinstance(block, dict) else (block.name if hasattr(block, 'name') else 'Unknown')
                        block_desc = block.get('description') if isinstance(block, dict) else (block.description if hasattr(block, 'description') else '')
                        block_content = block.get('content') if isinstance(block, dict) else (block.content if hasattr(block, 'content') else '')
                        
                        summary = "Block " + str(connected_id) + " (" + relation + "): " + block_name
                        # Add first 200 chars of description
                        if block_desc:
                            desc_preview = block_desc[:200] + "..." if len(block_desc) > 200 else block_desc
                            summary += "\n  Description: " + desc_preview
                        
                        # Add key functions/classes if identifiable
                        if block_content:
                            code_lines = block_content.split('\n')
                            key_items = []
                            for line in code_lines[:20]:  # Check first 20 lines
                                line = line.strip()
                                if line.startswith('def '):
                                    func_name = line.split('(')[0].replace('def ', '').strip()
                                    key_items.append("function: " + func_name)
                                elif line.startswith('class '):
                                    class_name = line.split('(')[0].split(':')[0].replace('class ', '').strip()
                                    key_items.append("class: " + class_name)
                            
                            if key_items:
                                summary += "\n  Key items: " + ', '.join(key_items[:5])  # Limit to 5 items
                        
                        connected_info.append(summary)
                        break
                except Exception as e:
                    print("Warning: Could not process block " + str(block) + ": " + str(e))
                    continue
    
    return "\n\n".join(connected_info) if connected_info else "No connected blocks."

# Pydantic models for component extraction
class Component(BaseModel):
    name: str = Field(description="Descriptive name for the component")
    number: int = Field(description="Sequential number within this block")
    type: str = Field(description="Component type: import, class, method, function, variable, expression, decorator")
    library: str = Field(description="Associated library or 'built-in'")
    code: str = Field(description="The exact code for this component")
    description: str = Field(description="Brief explanation of what this component does")
    parent: Optional[int] = Field(default=None, description="Parent component number")
    children: List[int] = Field(default_factory=list, description="Child component numbers")
    calls: List[int] = Field(default_factory=list, description="Component numbers this calls")
    called_by: List[int] = Field(default_factory=list, description="Component numbers that call this")

class ComponentAnalysis(BaseModel):
    components: List[Component] = Field(description="List of components found in the block")

# Function to analyze components within a block
@traceable
def analyze_block_components(
    block: dict,
    explanation: CodeExplanation,
    block_index: int
) -> ComponentAnalysis:
    """Analyze components within a specific code block"""
    
    # Create structured LLM for component analysis
    component_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    structured_component_llm = component_llm.with_structured_output(ComponentAnalysis)
    
    # Format connected blocks context
    connected_context = format_connected_blocks(
        explanation.edges,
        [b.model_dump() for b in explanation.blocks],
        block['id']
    )
    
    # Format the prompt
    formatted_prompt = COMPONENT_EXTRACTION_PROMPT.format(
        overview=explanation.overview,
        block_description=block['description'],
        block_id=block['id'],
        block_code=block['content'],
        connected_blocks_context=connected_context
    )
    
    # Create messages
    messages = [
        SystemMessage(content=formatted_prompt),
        HumanMessage(content="Extract all components from this code block.")
    ]
    
    # Get structured response
    result = structured_component_llm.invoke(messages)
    
    return result

# Example usage
# explanation = analyze_code('tiny_demo.ipynb')
# if explanation.blocks:
#     components = analyze_block_components(
#         explanation.blocks[0].model_dump(),
#         explanation,
#         0
#     )
#     print(json.dumps(components.model_dump(), indent=2))

#### Component Dependency Resolution

In [299]:
# Enhanced block analysis with dependency resolution
class BlockAnalysisState(BaseModel):
    """Track analysis state for multi-pass processing"""
    block_id: str
    initial_pass: bool = False
    is_complete: bool = False
    pending_dependencies: List[str] = Field(default_factory=list)  # Component references waiting for descriptions
    component_map: dict = Field(default_factory=dict)  # Map of component names to their descriptions

class EnhancedCodeExplanation(CodeExplanation):
    """Extended explanation with analysis states"""
    block_states: dict = Field(default_factory=dict)  # block_id -> BlockAnalysisState
    global_component_registry: dict = Field(default_factory=dict)  # "block_id:component_name" -> description

# Two-pass analysis system
@traceable
async def analyze_code_with_dependencies(ipynb_file: str) -> EnhancedCodeExplanation:
    """
    Analyze code with proper dependency resolution
    
    Phase 1: Initial block and component extraction
    Phase 2: Dependency resolution and description enhancement
    """
    
    # Phase 1: Initial analysis
    print("Phase 1: Initial block analysis...")
    initial_explanation = analyze_code(ipynb_file)
    enhanced_explanation = EnhancedCodeExplanation(**initial_explanation.model_dump())
    
    # Analyze each block's components
    for i, block in enumerate(enhanced_explanation.blocks):
        state = BlockAnalysisState(block_id=block.id)
        
        # Get components for this block
        components = analyze_block_components(
            block.model_dump(),
            enhanced_explanation,
            i
        )
        
        # Register components globally
        for comp in components.components:
            global_key = f"{block.id}:{comp.name}"
            enhanced_explanation.global_component_registry[global_key] = {
                "description": comp.description,
                "type": comp.type,
                "library": comp.library
            }
            state.component_map[comp.name] = comp
        
        # Identify pending dependencies
        state.pending_dependencies = identify_external_dependencies(
            components, 
            enhanced_explanation.edges,
            block.id
        )
        
        state.initial_pass = True
        state.is_complete = len(state.pending_dependencies) == 0
        enhanced_explanation.block_states[block.id] = state
    
    # Phase 2: Resolve dependencies
    print("Phase 2: Resolving dependencies...")
    incomplete_blocks = [
        block_id for block_id, state in enhanced_explanation.block_states.items()
        if state.initial_pass and not state.is_complete
    ]
    
    if incomplete_blocks:
        for block_id in incomplete_blocks:
            state = enhanced_explanation.block_states[block_id]
            block = next(b for b in enhanced_explanation.blocks if b.id == block_id)
            
            # Enhanced prompt with dependency context
            dependency_context = build_dependency_context(
                state.pending_dependencies,
                enhanced_explanation.global_component_registry
            )
            
            # Re-analyze with dependency context
            enhanced_components = await enhance_component_descriptions(
                block,
                state.component_map,
                dependency_context
            )
            
            # Update global registry with enhanced descriptions
            for comp in enhanced_components:
                global_key = f"{block_id}:{comp.name}"
                enhanced_explanation.global_component_registry[global_key]["description"] = comp.description
            
            state.is_complete = True
    
    print("Analysis complete!")
    return enhanced_explanation

def identify_external_dependencies(
    components: ComponentAnalysis,
    edges: List[Edge],
    current_block_id: str
) -> List[str]:
    """Identify components that depend on other blocks"""
    external_deps = []
    
    # Find blocks this block depends on
    dependent_block_ids = [
        edge.to for edge in edges 
        if edge.from_id == current_block_id
    ]
    
    for comp in components.components:
        # Check if component references external functions/classes
        # This is simplified - in reality you'd parse the code
        if comp.type in ["function", "class"] and dependent_block_ids:
            # Check if calls reference external blocks
            external_deps.append(f"{comp.name}:needs_context")
    
    return external_deps

def build_dependency_context(
    pending_deps: List[str],
    global_registry: dict
) -> str:
    """Build context string from resolved dependencies"""
    context_parts = []
    
    for dep in pending_deps:
        # Look up related components in global registry
        # This is simplified - you'd need to parse actual dependencies
        for key, info in global_registry.items():
            if info["type"] in ["function", "class"]:
                context_parts.append(
                    f"- {key}: {info['description'][:100]}..."
                )
    
    return "\n".join(context_parts)

async def enhance_component_descriptions(
    block: CodeBlock,
    component_map: dict,
    dependency_context: str
) -> List[Component]:
    """Re-analyze components with dependency context"""
    
    enhancement_prompt = """
    Given the following component and its dependencies, provide an enhanced description
    that explains how it uses the dependent components:
    
    Component: {component_name}
    Current Description: {current_description}
    
    Available Dependencies:
    {dependency_context}
    
    Provide an enhanced description that includes how this component interacts with its dependencies.
    """
    
    # This is where you'd call the LLM with the enhancement prompt
    # For now, returning the original components
    return list(component_map.values())

# 📚 STEP 4: Deep Description Generation with Iterative Improvement

This step enhances the analysis through:
- **Deep Description Generation**: Creates comprehensive 3-5 paragraph explanations for each block
- **Iterative Improvement**: Uses a judging agent to assess and selectively improve descriptions
- **Component Enhancement**: Updates component descriptions with architectural context
- **Smart Updates**: Only regenerates content that would meaningfully benefit

The system prevents wasteful API calls through intelligent improvement assessment and selective updates.

### Iterative Improvement System

This phase implements an intelligent iterative improvement system that:
1. Generates a comprehensive overview from deep descriptions
2. Uses a judging agent to assess improvement potential (0-100%)
3. Selectively updates only components that would benefit (>10% threshold)
4. Prevents wasteful regeneration through smart cascading
5. Supports up to 2 iterations with early stopping

Key features:
- **Improvement Assessment**: Measures semantic change and new insights
- **Selective Updates**: Only regenerates blocks/components that need it
- **Cost Optimization**: Minimizes API calls through targeted updates
- **Dependency Awareness**: Updates cascade only when meaningful changes occur

In [300]:
# Iterative Improvement System
from typing import List, Dict, Optional, Set
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langsmith import traceable
import json
import asyncio

# Models for improvement assessment
class ImprovementAssessment(BaseModel):
    """Assessment of potential improvements from iteration"""
    improvement_score: float = Field(description="Overall improvement potential (0-100)")
    critical_insights: List[str] = Field(default_factory=list, description="New important discoveries")
    blocks_needing_update: List[str] = Field(default_factory=list, description="Block IDs that would benefit from updates")
    components_needing_context: List[str] = Field(default_factory=list, description="Component IDs needing more context")
    rationale: str = Field(description="Explanation of the assessment")

class UpdatePlan(BaseModel):
    """Plan for selective updates based on assessment"""
    update_overview: bool = False
    blocks_to_update: List[str] = Field(default_factory=list)
    components_to_update: List[str] = Field(default_factory=list)
    requires_cascade: bool = False
    estimated_cost: str = Field(default="low", description="low/medium/high")

# Prompts for improvement system
COMPREHENSIVE_OVERVIEW_PROMPT = """You are creating a comprehensive overview that synthesizes all the deep insights from analyzing a codebase.

## Original Overview:
{original_overview}

## Deep Block Descriptions:
{deep_descriptions}

## Task:
Create an enhanced overview that:
1. Incorporates key insights from the deep analysis
2. Highlights important architectural patterns discovered
3. Explains critical relationships between components
4. Maintains clarity while adding depth

Provide a 3-4 paragraph comprehensive overview that captures the full understanding of the codebase.
"""

IMPROVEMENT_JUDGE_PROMPT = """You are an expert code analysis judge evaluating whether further iteration would provide meaningful value.

## Original Analysis:
**Overview**: {original_overview}
**Block Count**: {block_count}

## After Deep Analysis:
**Comprehensive Overview**: {comprehensive_overview}

## Deep Descriptions Summary:
{deep_descriptions_summary}

## Critical Questions:
1. What NEW architectural insights were discovered in the deep analysis?
2. Which blocks revealed unexpected complexity or dependencies?
3. What cross-block relationships became clearer?
4. Where would additional iteration provide significant value?

## Assessment Task:
Provide a JSON response with:
- improvement_score: (0-100) representing the potential value of another iteration
- critical_insights: List of new important discoveries
- blocks_needing_update: List of block IDs that would benefit from reanalysis
- components_needing_context: Specific components that need more context
- rationale: Explanation of your assessment

Score Guidelines:
- 0-10: Minimal new insights, well understood
- 10-30: Some new insights, moderate complexity revealed
- 30-50: Significant new patterns discovered
- 50+: Major architectural insights, high value in iteration
"""

COMPONENT_UPDATE_PROMPT = """You are enhancing a component's description with full architectural understanding.

## Comprehensive Architecture Overview:
{comprehensive_overview}

## Block Context:
**Block Name**: {block_name}
**Block Description**: {block_description}

## Component Details:
- **Name**: {component_name}
- **Type**: {component_type}
- **Current Description**: {current_description}
- **Related Components**: {related_components}

## Task:
Provide an enhanced description that:
1. Explains this component's role in the larger architecture
2. Clarifies how it contributes to the block's purpose
3. Highlights its interactions with other components
4. Incorporates insights from the comprehensive analysis

Generate a concise but insightful description (2-3 sentences) that captures the component's significance.
"""

# Core improvement functions
@traceable
async def generate_comprehensive_overview(
    walkthrough_state: WalkthroughState
) -> str:
    """Generate a comprehensive overview incorporating deep descriptions"""
    
    # Format deep descriptions
    deep_desc_text = []
    for block_id, block_state in walkthrough_state.block_states.items():
        if block_state.deep_description:
            deep_desc_text.append(
                f"Block {block_id} ({block_state.block_name}):\n{block_state.deep_description[:500]}..."
            )
    
    formatted_prompt = COMPREHENSIVE_OVERVIEW_PROMPT.format(
        original_overview=walkthrough_state.code_overview,
        deep_descriptions="\n\n".join(deep_desc_text)
    )
    
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
    messages = [
        SystemMessage(content=formatted_prompt),
        HumanMessage(content="Generate the comprehensive overview.")
    ]
    
    response = await llm.ainvoke(messages)
    return response.content

@traceable
async def judge_improvement_potential(
    original_overview: str,
    comprehensive_overview: str,
    walkthrough_state: WalkthroughState
) -> ImprovementAssessment:
    """Judge whether iteration would provide meaningful improvement"""
    
    # Create summary of deep descriptions
    deep_summary = []
    for block_id, block_state in walkthrough_state.block_states.items():
        if block_state.deep_description:
            # Extract key points from deep description
            desc_preview = block_state.deep_description[:200]
            deep_summary.append(f"- Block {block_id}: {desc_preview}...")
    
    formatted_prompt = IMPROVEMENT_JUDGE_PROMPT.format(
        original_overview=original_overview,
        block_count=len(walkthrough_state.block_states),
        comprehensive_overview=comprehensive_overview,
        deep_descriptions_summary="\n".join(deep_summary)
    )
    
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    structured_llm = llm.with_structured_output(ImprovementAssessment)
    
    messages = [
        SystemMessage(content=formatted_prompt),
        HumanMessage(content="Assess the improvement potential.")
    ]
    
    return await structured_llm.ainvoke(messages)

def create_update_plan(
    assessment: ImprovementAssessment,
    current_state: WalkthroughState
) -> UpdatePlan:
    """Create an efficient update plan based on assessment"""
    plan = UpdatePlan()
    
    # Decide if overview needs update
    if assessment.critical_insights:
        plan.update_overview = True
    
    # Add blocks that need updates
    plan.blocks_to_update = assessment.blocks_needing_update
    
    # Identify components affected by block updates
    affected_components = set()
    for block_id in plan.blocks_to_update:
        if block_id in current_state.block_states:
            block_state = current_state.block_states[block_id]
            # Add components from this block
            affected_components.update(block_state.component_states.keys())
            # Add dependent components
            for dep_block_id in block_state.depended_by_blocks:
                if dep_block_id in current_state.block_states:
                    dep_block = current_state.block_states[dep_block_id]
                    affected_components.update(dep_block.component_states.keys())
    
    plan.components_to_update = list(affected_components)
    
    # Estimate cost
    total_updates = len(plan.blocks_to_update) + len(plan.components_to_update)
    if total_updates > 10:
        plan.estimated_cost = "high"
    elif total_updates > 5:
        plan.estimated_cost = "medium"
    else:
        plan.estimated_cost = "low"
    
    # Check if cascade needed
    plan.requires_cascade = len(plan.components_to_update) > len(plan.blocks_to_update) * 2
    
    return plan

# Selective update functions
@traceable
async def update_specific_blocks(
    walkthrough_state: WalkthroughState,
    block_ids: List[str],
    comprehensive_overview: str
) -> Dict[str, str]:
    """Update only specific blocks with new context"""
    updated_descriptions = {}
    
    update_prompt = """Given the comprehensive understanding of the codebase, provide an enhanced description for this block.

Comprehensive Overview:
{comprehensive_overview}

Current Block: {block_name}
Current Description:
{current_description}

Related Insights:
{related_insights}

Generate an enhanced description that incorporates the new architectural understanding.
"""
    
    for block_id in block_ids:
        if block_id not in walkthrough_state.block_states:
            continue
            
        block_state = walkthrough_state.block_states[block_id]
        
        # Gather related insights
        related = []
        for dep_id in block_state.depends_on_blocks + block_state.depended_by_blocks:
            if dep_id in walkthrough_state.block_states:
                dep_block = walkthrough_state.block_states[dep_id]
                related.append(f"- {dep_block.block_name}: {dep_block.deep_description[:100]}...")
        
        formatted_prompt = update_prompt.format(
            comprehensive_overview=comprehensive_overview,
            block_name=block_state.block_name,
            current_description=block_state.deep_description or block_state.initial_description,
            related_insights="\n".join(related) if related else "None"
        )
        
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
        messages = [
            SystemMessage(content=formatted_prompt),
            HumanMessage(content="Generate the enhanced block description.")
        ]
        
        response = await llm.ainvoke(messages)
        updated_descriptions[block_id] = response.content
        
        # Update the state
        block_state.deep_description = response.content
    
    return updated_descriptions

@traceable
async def update_components_with_context(
    walkthrough_state: WalkthroughState,
    component_ids: List[str],
    comprehensive_overview: str
) -> Dict[str, str]:
    """Update component descriptions with full architectural context"""
    updated_components = {}
    
    # Process in batches for efficiency
    for comp_id in component_ids:
        block_id = comp_id.split(":")[0]
        if block_id not in walkthrough_state.block_states:
            continue
            
        block_state = walkthrough_state.block_states[block_id]
        if comp_id not in block_state.component_states:
            continue
            
        comp_state = block_state.component_states[comp_id]
        
        # Find related components in the same block
        related_comps = []
        for other_comp_id, other_comp in block_state.component_states.items():
            if other_comp_id != comp_id:
                # Check if this component is related
                # Use the correct attribute names for ComponentState
                if (comp_id in other_comp.calls_component_ids or 
                    comp_id in other_comp.called_by_component_ids or
                    comp_id in other_comp.parent_component_ids or
                    comp_id in other_comp.child_component_ids):
                    related_comps.append(f"{other_comp.component_name} ({other_comp.type})")
        
        formatted_prompt = COMPONENT_UPDATE_PROMPT.format(
            comprehensive_overview=comprehensive_overview,
            block_name=block_state.block_name,
            block_description=block_state.deep_description or block_state.initial_description,
            component_name=comp_state.component_name,
            component_type=comp_state.type,
            current_description=comp_state.enhanced_description or comp_state.description,
            related_components=", ".join(related_comps) if related_comps else "None identified"
        )
        
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
        messages = [
            SystemMessage(content=formatted_prompt),
            HumanMessage(content="Generate the enhanced component description.")
        ]
        
        response = await llm.ainvoke(messages)
        enhanced_desc = response.content
        
        # Update the component state
        comp_state.enhanced_description = enhanced_desc
        updated_components[comp_id] = enhanced_desc
        
        # Update global registry
        if comp_id in walkthrough_state.global_component_registry:
            walkthrough_state.global_component_registry[comp_id]["description"] = enhanced_desc
    
    return updated_components

# Main iterative improvement function
@traceable
async def smart_iterative_improvement(
    walkthrough_state: WalkthroughState,
    max_iterations: int = 2,
    improvement_threshold: float = 10.0,
    update_components: bool = True
) -> WalkthroughState:
    """Intelligent iterative improvement with waste prevention"""
    
    print("\n=== STARTING ITERATIVE IMPROVEMENT ===")
    iteration = 0
    improvement_history = []
    total_api_calls = 0
    
    while iteration < max_iterations:
        print(f"\n--- Iteration {iteration + 1} ---")
        
        # 1. Generate comprehensive overview from deep descriptions
        print("Generating comprehensive overview...")
        comprehensive_overview = await generate_comprehensive_overview(walkthrough_state)
        total_api_calls += 1
        
        # 2. Assess improvement potential
        print("Assessing improvement potential...")
        assessment = await judge_improvement_potential(
            original_overview=walkthrough_state.code_overview,
            comprehensive_overview=comprehensive_overview,
            walkthrough_state=walkthrough_state
        )
        total_api_calls += 1
        
        print(f"Improvement score: {assessment.improvement_score:.1f}%")
        print(f"Rationale: {assessment.rationale}")
        
        # 3. Stop if minimal improvement
        if assessment.improvement_score < improvement_threshold:
            print(f"Below {improvement_threshold}% threshold, stopping iteration")
            break
        
        # 4. Create update plan
        update_plan = create_update_plan(assessment, walkthrough_state)
        print(f"\nUpdate plan:")
        print(f"  - Update overview: {update_plan.update_overview}")
        print(f"  - Blocks to update: {len(update_plan.blocks_to_update)}")
        print(f"  - Components affected: {len(update_plan.components_to_update)}")
        print(f"  - Estimated cost: {update_plan.estimated_cost}")
        
        # 5. Execute updates efficiently
        if update_plan.update_overview:
            print("Updating overview with new insights...")
            walkthrough_state.code_overview = comprehensive_overview
        
        if update_plan.blocks_to_update:
            print(f"Updating {len(update_plan.blocks_to_update)} blocks...")
            updated_blocks = await update_specific_blocks(
                walkthrough_state,
                update_plan.blocks_to_update,
                comprehensive_overview
            )
            total_api_calls += len(update_plan.blocks_to_update)
            
        # 6. Component updates
        if update_components and update_plan.components_to_update:
            print(f"Updating {len(update_plan.components_to_update)} components with architectural context...")
            updated_comps = await update_components_with_context(
                walkthrough_state,
                update_plan.components_to_update,
                comprehensive_overview
            )
            total_api_calls += len(update_plan.components_to_update)
            print(f"  ✓ Updated {len(updated_comps)} component descriptions")
        
        improvement_history.append(assessment.improvement_score)
        iteration += 1
        
        # Prevent oscillation
        if len(improvement_history) > 1 and improvement_history[-1] <= improvement_history[-2]:
            print("Improvement plateaued, stopping")
            break
    
    # Final component update pass if we haven't done it yet
    if update_components and iteration > 0:
        # Find any components that haven't been updated
        all_component_ids = []
        for block_state in walkthrough_state.block_states.values():
            all_component_ids.extend(block_state.component_states.keys())
        
        # Check which components need the architectural context
        components_to_finalize = []
        for comp_id in all_component_ids:
            block_id = comp_id.split(":")[0]
            if block_id in walkthrough_state.block_states:
                comp_state = walkthrough_state.block_states[block_id].component_states.get(comp_id)
                if comp_state and not comp_state.enhanced_description:
                    components_to_finalize.append(comp_id)
        
        if components_to_finalize:
            print(f"\nFinal pass: Updating {len(components_to_finalize)} remaining components...")
            final_updates = await update_components_with_context(
                walkthrough_state,
                components_to_finalize,
                walkthrough_state.code_overview
            )
            total_api_calls += len(components_to_finalize)
            print(f"  ✓ Updated {len(final_updates)} component descriptions")
    
    print(f"\n=== IMPROVEMENT COMPLETE ===")
    print(f"Total iterations: {iteration}")
    print(f"Total API calls: {total_api_calls}")
    print(f"Improvement history: {improvement_history}")
    
    return walkthrough_state

# Example usage in LangGraph
async def improvement_node(state: WalkthroughState) -> WalkthroughState:
    """LangGraph node for iterative improvement"""
    state.analysis_phase = "improvement"
    
    # Only run improvement if all blocks have deep descriptions
    if all(bs.deep_description for bs in state.block_states.values()):
        improved_state = await smart_iterative_improvement(state, update_components=True)
        state.messages.append("Completed iterative improvement analysis")
        return improved_state
    
    return state

In [301]:
# LangGraph State Definitions for Multi-Pass Analysis
from typing import Dict, List, Optional, Literal
from pydantic import BaseModel, Field
from datetime import datetime

# Component-level state
class ComponentState(BaseModel):
    """State for individual code components"""
    # Basic info
    component_id: str = Field(description="Unique ID: block_id:component_number")
    component_name: str
    component_number: int
    type: Literal["import", "class", "method", "function", "variable", "expression", "decorator"]
    library: str
    
    # Code and description
    full_code: str
    description: str
    enhanced_description: Optional[str] = None  # After dependency resolution
    
    # Location info
    block_id: str  # Block it belongs to
    line_start: int
    line_end: int
    
    # Relationships
    parent_component_ids: List[str] = Field(default_factory=list)
    child_component_ids: List[str] = Field(default_factory=list)
    calls_component_ids: List[str] = Field(default_factory=list)  # What this calls
    called_by_component_ids: List[str] = Field(default_factory=list)  # What calls this
    
    # Analysis state
    external_dependencies: List[str] = Field(default_factory=list)  # Dependencies in other blocks
    is_resolved: bool = False  # True when all dependencies are resolved
    confidence_score: float = 1.0  # 0-1, how confident the analysis is

# Block-level state
class BlockState(BaseModel):
    """State for code blocks"""
    # Basic info
    block_id: str
    block_name: str
    block_type: Literal["imports", "utilities", "classes", "main", "config", "mixed"]
    
    # Content
    block_content: str  # The actual code
    initial_description: str
    deep_description: Optional[str] = None  # Enhanced description after analysis
    
    # Components
    component_states: Dict[str, ComponentState] = Field(default_factory=dict)  # component_id -> ComponentState
    
    # Analysis progress
    initial_pass_complete: bool = False
    dependency_resolution_complete: bool = False
    is_fully_analyzed: bool = False  # True when all analysis is done
    
    # Dependencies
    depends_on_blocks: List[str] = Field(default_factory=list)  # Block IDs this depends on
    depended_by_blocks: List[str] = Field(default_factory=list)  # Block IDs that depend on this
    pending_dependencies: List[str] = Field(default_factory=list)  # Unresolved dependencies
    
    # Metadata
    execution_order_index: Optional[int] = None  # Suggested order for understanding
    last_updated: datetime = Field(default_factory=datetime.now)

# Top-level walkthrough state
class WalkthroughState(BaseModel):
    """Complete state for code walkthrough"""
    # Overview
    code_overview: str  # High-level summary of entire codebase
    notebook_path: str
    total_blocks: int = 0
    total_components: int = 0
    
    # Blocks
    block_states: Dict[str, BlockState] = Field(default_factory=dict)  # block_id -> BlockState
    
    # Global registries
    global_component_registry: Dict[str, Dict] = Field(default_factory=dict)  # component_id -> {name, type, description}
    import_registry: Dict[str, List[str]] = Field(default_factory=dict)  # library -> [component_ids]
    function_registry: Dict[str, str] = Field(default_factory=dict)  # function_name -> component_id
    class_registry: Dict[str, str] = Field(default_factory=dict)  # class_name -> component_id
    
    # Analysis flow
    analysis_phase: Literal["initializing", "block_analysis", "component_extraction", "dependency_resolution", "enhancement", "complete"] = "initializing"
    execution_order: List[str] = Field(default_factory=list)  # Suggested block reading order
    dependency_graph: Dict[str, List[str]] = Field(default_factory=dict)  # block_id -> [dependent_block_ids]
    
    # Progress tracking
    blocks_analyzed: int = 0
    blocks_with_dependencies: int = 0
    dependencies_resolved: int = 0
    
    # Metadata
    created_at: datetime = Field(default_factory=datetime.now)
    last_updated: datetime = Field(default_factory=datetime.now)
    analysis_config: dict = Field(default_factory=dict)  # Store config like model, temperature, etc.
    
    # LangGraph specific
    messages: List[str] = Field(default_factory=list)  # Status messages
    errors: List[str] = Field(default_factory=list)  # Error messages
    
    def get_progress(self) -> dict:
        """Get analysis progress summary"""
        return {
            "phase": self.analysis_phase,
            "blocks_analyzed": f"{self.blocks_analyzed}/{self.total_blocks}",
            "dependencies_resolved": f"{self.dependencies_resolved}/{self.blocks_with_dependencies}",
            "is_complete": self.analysis_phase == "complete"
        }
    
    def get_unresolved_blocks(self) -> List[str]:
        """Get blocks that still need dependency resolution"""
        return [
            block_id for block_id, state in self.block_states.items()
            if state.initial_pass_complete and not state.is_fully_analyzed
        ]
    
    def update_component_description(self, component_id: str, enhanced_description: str):
        """Update a component's description after dependency resolution"""
        block_id, comp_num = component_id.split(":")
        if block_id in self.block_states:
            if component_id in self.block_states[block_id].component_states:
                self.block_states[block_id].component_states[component_id].enhanced_description = enhanced_description
                self.block_states[block_id].component_states[component_id].is_resolved = True
                
    def mark_block_complete(self, block_id: str):
        """Mark a block as fully analyzed"""
        if block_id in self.block_states:
            self.block_states[block_id].is_fully_analyzed = True
            self.block_states[block_id].dependency_resolution_complete = True
            self.dependencies_resolved += 1

In [302]:
# Deep Description Generation Phase
from typing import List, Dict, Optional
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langsmith import traceable
import json

# Prompt for generating comprehensive deep descriptions
DEEP_DESCRIPTION_PROMPT = """You are an expert code educator creating a comprehensive description of a code block.

## Context:
**Overall Code Purpose**: {code_overview}
**Block Name**: {block_name}
**Block Type**: {block_type}
**Initial Description**: {initial_description}

## Components in this block:
{components_summary}

## Dependencies:
**This block depends on**: {depends_on}
**Other blocks depend on this**: {depended_by}

## Task:
Generate a deep, educational description that:
1. Explains the block's overall purpose and role in the codebase
2. Describes how the components work together
3. Highlights key implementation details
4. Explains interactions with other blocks
5. Provides insights into the design decisions

## Output Format:
Provide a comprehensive narrative description (3-5 paragraphs) that would help someone understand:
- WHAT this block does
- HOW it accomplishes its goals
- WHY it's structured this way
- WHEN it gets used in the execution flow
"""

@traceable
async def generate_deep_description(
    block_state: BlockState,
    walkthrough_state: WalkthroughState,
    components_with_descriptions: Dict[str, str]
) -> str:
    """
    Generate a comprehensive deep description for a fully analyzed block.
    
    Args:
        block_state: The block's current state with all components
        walkthrough_state: Global state for context
        components_with_descriptions: Map of component_id to final description
        
    Returns:
        Deep description narrative
    """
    # Format components summary
    components_lines = []
    for comp_id, comp_state in block_state.component_states.items():
        # Use enhanced description if available, otherwise initial
        description = components_with_descriptions.get(
            comp_id, 
            comp_state.enhanced_description or comp_state.description
        )
        components_lines.append(
            f"- **{comp_state.component_name}** ({comp_state.type}): {description}"
        )
    components_summary = "\n".join(components_lines)
    
    # Get dependency information
    depends_on_names = []
    for dep_id in block_state.depends_on_blocks:
        if dep_id in walkthrough_state.block_states:
            depends_on_names.append(walkthrough_state.block_states[dep_id].block_name)
    
    depended_by_names = []
    for dep_id in block_state.depended_by_blocks:
        if dep_id in walkthrough_state.block_states:
            depended_by_names.append(walkthrough_state.block_states[dep_id].block_name)
    
    # Format the prompt
    formatted_prompt = DEEP_DESCRIPTION_PROMPT.format(
        code_overview=walkthrough_state.code_overview,
        block_name=block_state.block_name,
        block_type=block_state.block_type,
        initial_description=block_state.initial_description,
        components_summary=components_summary,
        depends_on=", ".join(depends_on_names) if depends_on_names else "None",
        depended_by=", ".join(depended_by_names) if depended_by_names else "None"
    )
    
    # Call LLM
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)  # Slightly higher temp for narrative
    
    messages = [
        SystemMessage(content=formatted_prompt),
        HumanMessage(content="Generate the comprehensive deep description for this block.")
    ]
    
    response = await llm.ainvoke(messages)
    
    return response.content

# Batch processing for multiple blocks
@traceable
async def generate_all_deep_descriptions(
    walkthrough_state: WalkthroughState,
    blocks_to_process: Optional[List[str]] = None
) -> Dict[str, str]:
    """
    Generate deep descriptions for all specified blocks.
    
    Args:
        walkthrough_state: Current analysis state
        blocks_to_process: List of block IDs to process (None = all blocks)
        
    Returns:
        Dictionary mapping block_id to deep_description
    """
    deep_descriptions = {}
    
    # Determine which blocks to process
    if blocks_to_process is None:
        blocks_to_process = [
            block_id for block_id, block_state in walkthrough_state.block_states.items()
            if block_state.is_fully_analyzed and not block_state.deep_description
        ]
    
    print(f"Generating deep descriptions for {len(blocks_to_process)} blocks...")
    
    for block_id in blocks_to_process:
        block_state = walkthrough_state.block_states[block_id]
        
        # Prepare final component descriptions
        components_with_descriptions = {}
        for comp_id, comp_state in block_state.component_states.items():
            # Prioritize enhanced descriptions
            final_desc = comp_state.enhanced_description or comp_state.description
            components_with_descriptions[comp_id] = final_desc
        
        # Generate deep description
        deep_desc = await generate_deep_description(
            block_state,
            walkthrough_state,
            components_with_descriptions
        )
        
        deep_descriptions[block_id] = deep_desc
        
        # Update the block state
        block_state.deep_description = deep_desc
        
        print(f"  ✓ Generated deep description for block {block_id}: {block_state.block_name}")
    
    return deep_descriptions

# Integration with branching logic
async def process_block_with_branching(
    block_id: str,
    walkthrough_state: WalkthroughState
) -> str:
    """
    Process a block through the appropriate branch based on its completion status.
    
    This implements the branching logic:
    - is_completed=true → straight to deep description
    - is_completed=false → dependency resolution → deep description
    """
    block_state = walkthrough_state.block_states[block_id]
    
    if block_state.is_fully_analyzed and block_state.deep_description:
        # Already processed
        return block_state.deep_description
    
    # Check if block needs dependency resolution
    if not block_state.dependency_resolution_complete:
        # This block needs dependency resolution first
        print(f"Block {block_id} requires dependency resolution first")
        # (This would be handled by the resolve_dependencies node)
        return None
    
    # Block is ready for deep description generation
    components_with_descriptions = {}
    for comp_id, comp_state in block_state.component_states.items():
        final_desc = comp_state.enhanced_description or comp_state.description
        components_with_descriptions[comp_id] = final_desc
    
    deep_desc = await generate_deep_description(
        block_state,
        walkthrough_state,
        components_with_descriptions
    )
    
    return deep_desc

# Example usage in LangGraph node
async def deep_description_node(state: WalkthroughState) -> WalkthroughState:
    """LangGraph node for deep description generation"""
    state.analysis_phase = "enhancement"
    
    # Find blocks ready for deep descriptions
    ready_blocks = [
        block_id for block_id, block_state in state.block_states.items()
        if block_state.is_fully_analyzed and not block_state.deep_description
    ]
    
    if ready_blocks:
        deep_descriptions = await generate_all_deep_descriptions(state, ready_blocks)
        state.messages.append(f"Generated {len(deep_descriptions)} deep descriptions")
    
    return state

### LangGraph State Definitions for Multi-Pass Analysis

### LangGraph Node Definitions

In [303]:
# LangGraph Node Definitions
from langgraph.graph import StateGraph, END
from typing import Any, Dict

# Node 1: Initialize and extract blocks
async def initialize_analysis(state: WalkthroughState) -> WalkthroughState:
    """Initialize analysis and extract code blocks"""
    state.analysis_phase = "block_analysis"
    state.messages.append("Starting code analysis...")
    
    # Extract code from notebook (will also write .py and .md files as side effect)
    python_code, _ = separate_ipynb(state.notebook_path)
    
    # Get initial block breakdown
    explanation = analyze_code(state.notebook_path)
    
    # Populate state with blocks
    state.code_overview = explanation.overview
    state.total_blocks = len(explanation.blocks)
    state.execution_order = explanation.execution_order
    
    for block in explanation.blocks:
        block_state = BlockState(
            block_id=block.id,
            block_name=block.name,
            block_type=determine_block_type(block.content),
            block_content=block.content,
            initial_description=block.description,
            depends_on_blocks=block.dependencies,
            depended_by_blocks=block.dependents
        )
        state.block_states[block.id] = block_state
    
    # Build dependency graph
    for edge in explanation.edges:
        # Access the from field using the actual attribute name (not the alias)
        from_id = edge.from_id  # This is the actual attribute name
        to_id = edge.to
        if from_id not in state.dependency_graph:
            state.dependency_graph[from_id] = []
        state.dependency_graph[from_id].append(to_id)
    
    state.messages.append(f"Extracted {state.total_blocks} blocks")
    return state

# Node 2: Extract components from blocks
async def extract_components(state: WalkthroughState) -> WalkthroughState:
    """Extract components from each block"""
    state.analysis_phase = "component_extraction"
    
    # First, create a minimal CodeExplanation object from state for analyze_block_components
    blocks = []
    edges = []
    
    for block_id, block_state in state.block_states.items():
        block = CodeBlock(
            id=block_id,
            name=block_state.block_name,
            description=block_state.initial_description,
            content=block_state.block_content,
            dependencies=block_state.depends_on_blocks,
            dependents=block_state.depended_by_blocks
        )
        blocks.append(block)
    
    # Create edges from dependency graph - note we use 'from' not 'from_id'
    for from_id, to_ids in state.dependency_graph.items():
        for to_id in to_ids:
            edge = Edge(**{"from": from_id, "to": to_id, "type": "uses"})
            edges.append(edge)
    
    explanation = CodeExplanation(
        overview=state.code_overview,
        blocks=blocks,
        edges=edges,
        execution_order=state.execution_order
    )
    
    for block_id, block_state in state.block_states.items():
        if not block_state.initial_pass_complete:
            # Create a dictionary for analyze_block_components with the correct field names
            block_dict = {
                'id': block_id,  # Use block_id for 'id'
                'name': block_state.block_name,
                'description': block_state.initial_description,
                'content': block_state.block_content,
                'dependencies': block_state.depends_on_blocks,
                'dependents': block_state.depended_by_blocks
            }
            
            # Extract components for this block
            components = analyze_block_components(
                block_dict,
                explanation,
                list(state.block_states.keys()).index(block_id)
            )
            
            # Populate component states
            for comp in components.components:
                # Handle parent field - it might be an int or None
                parent_ids = []
                if comp.parent is not None:
                    if isinstance(comp.parent, int):
                        parent_ids = [f"{block_id}:{comp.parent}"]
                    elif isinstance(comp.parent, list):
                        parent_ids = [f"{block_id}:{p}" for p in comp.parent]
                
                comp_state = ComponentState(
                    component_id=f"{block_id}:{comp.number}",
                    component_name=comp.name,
                    component_number=comp.number,
                    type=comp.type,
                    library=comp.library,
                    full_code=comp.code,
                    description=comp.description,
                    block_id=block_id,
                    line_start=0,  # Would need AST parsing for accurate lines
                    line_end=0,
                    parent_component_ids=parent_ids,
                    child_component_ids=[f"{block_id}:{c}" for c in comp.children],
                    calls_component_ids=[f"{block_id}:{c}" for c in comp.calls],
                    called_by_component_ids=[f"{block_id}:{c}" for c in comp.called_by]
                )
                
                # Check for external dependencies
                if block_state.depends_on_blocks:
                    comp_state.external_dependencies = identify_external_refs(
                        comp.code, 
                        block_state.depends_on_blocks,
                        state
                    )
                
                block_state.component_states[comp_state.component_id] = comp_state
                
                # Update global registries
                state.global_component_registry[comp_state.component_id] = {
                    "name": comp.name,
                    "type": comp.type,
                    "description": comp.description,
                    "library": comp.library
                }
                
                # Update specific registries
                if comp.type == "function":
                    state.function_registry[comp.name] = comp_state.component_id
                elif comp.type == "class":
                    state.class_registry[comp.name] = comp_state.component_id
                elif comp.type == "import":
                    if comp.library not in state.import_registry:
                        state.import_registry[comp.library] = []
                    state.import_registry[comp.library].append(comp_state.component_id)
                
                state.total_components += 1
            
            block_state.initial_pass_complete = True
            state.blocks_analyzed += 1
            
            # Check if block needs dependency resolution
            if any(comp_state.external_dependencies for comp_state in block_state.component_states.values()):
                block_state.pending_dependencies = [
                    comp_id for comp_id, comp_state in block_state.component_states.items()
                    if comp_state.external_dependencies
                ]
                state.blocks_with_dependencies += 1
    
    state.messages.append(f"Extracted {state.total_components} components from {state.blocks_analyzed} blocks")
    return state

# Node 3: Resolve dependencies
async def resolve_dependencies(state: WalkthroughState) -> WalkthroughState:
    """Resolve cross-block dependencies"""
    state.analysis_phase = "dependency_resolution"
    
    unresolved_blocks = state.get_unresolved_blocks()
    if not unresolved_blocks:
        state.messages.append("No dependencies to resolve")
        return state
    
    state.messages.append(f"Resolving dependencies for {len(unresolved_blocks)} blocks")
    
    for block_id in unresolved_blocks:
        block_state = state.block_states[block_id]
        
        # Gather context from dependencies
        dependency_context = {}
        for comp_id, comp_state in block_state.component_states.items():
            if comp_state.external_dependencies:
                for ext_dep in comp_state.external_dependencies:
                    # Look up the external component
                    if ext_dep in state.global_component_registry:
                        dependency_context[ext_dep] = state.global_component_registry[ext_dep]
        
        # Enhance descriptions with context
        if dependency_context:
            enhanced_descriptions = await enhance_with_dependencies(
                block_state,
                dependency_context,
                state
            )
            
            # Update component descriptions
            for comp_id, enhanced_desc in enhanced_descriptions.items():
                state.update_component_description(comp_id, enhanced_desc)
        
        state.mark_block_complete(block_id)
    
    state.messages.append(f"Resolved dependencies for {len(unresolved_blocks)} blocks")
    return state

# Node 4: Finalize analysis
async def finalize_analysis(state: WalkthroughState) -> WalkthroughState:
    """Finalize the analysis and prepare output"""
    state.analysis_phase = "complete"
    state.last_updated = datetime.now()
    
    # Generate enhanced block descriptions
    for block_id, block_state in state.block_states.items():
        if block_state.is_fully_analyzed and not block_state.deep_description:
            # Combine component insights for deep description
            component_insights = []
            for comp_state in block_state.component_states.values():
                desc = comp_state.enhanced_description or comp_state.description
                component_insights.append(f"- {comp_state.component_name}: {desc}")
            
            block_state.deep_description = (
                f"{block_state.initial_description}\n\n"
                f"Components:\n" + "\n".join(component_insights)
            )
    
    state.messages.append("Analysis complete!")
    return state

# Helper functions
def determine_block_type(content: str) -> str:
    """Determine the type of a code block"""
    if "import " in content or "from " in content:
        return "imports"
    elif "class " in content:
        return "classes"
    elif "def " in content and "class " not in content:
        return "utilities"
    elif "__name__" in content or "main()" in content:
        return "main"
    elif any(var in content for var in ["CONFIG", "SETTINGS", "PARAMS"]):
        return "config"
    else:
        return "mixed"

def identify_external_refs(code: str, dependent_blocks: List[str], state: WalkthroughState) -> List[str]:
    """Identify references to components in other blocks"""
    external_refs = []
    
    # Simple pattern matching - in production would use AST
    for func_name, comp_id in state.function_registry.items():
        if func_name + "(" in code:
            block_id = comp_id.split(":")[0]
            if block_id in dependent_blocks:
                external_refs.append(comp_id)
    
    for class_name, comp_id in state.class_registry.items():
        if class_name + "(" in code or class_name + "." in code:
            block_id = comp_id.split(":")[0]
            if block_id in dependent_blocks:
                external_refs.append(comp_id)
    
    return external_refs

# Build the graph
def create_analysis_graph():
    """Create the LangGraph workflow"""
    workflow = StateGraph(WalkthroughState)
    
    # Add nodes
    workflow.add_node("initialize", initialize_analysis)
    workflow.add_node("extract_components", extract_components)
    workflow.add_node("resolve_dependencies", resolve_dependencies)
    workflow.add_node("finalize", finalize_analysis)
    
    # Define flow
    workflow.add_edge("initialize", "extract_components")
    workflow.add_edge("extract_components", "resolve_dependencies")
    workflow.add_edge("resolve_dependencies", "finalize")
    workflow.add_edge("finalize", END)
    
    # Set entry point
    workflow.set_entry_point("initialize")
    
    return workflow.compile()

# Usage example
# graph = create_analysis_graph()
# initial_state = WalkthroughState(notebook_path="example.ipynb")
# final_state = await graph.ainvoke(initial_state)

### State Flow Visualization

```mermaid
stateDiagram-v2
    [*] --> Initialize: WalkthroughState(notebook_path)
    
    Initialize --> ExtractComponents: Block extraction complete
    state Initialize {
        [*] --> AnalyzeNotebook
        AnalyzeNotebook --> PopulateBlocks
        PopulateBlocks --> BuildDependencyGraph
    }
    
    ExtractComponents --> ResolveDependencies: Components extracted
    state ExtractComponents {
        [*] --> ProcessBlock
        ProcessBlock --> ExtractBlockComponents
        ExtractBlockComponents --> UpdateRegistries
        UpdateRegistries --> CheckDependencies
        CheckDependencies --> ProcessBlock: More blocks
        CheckDependencies --> [*]: All blocks done
    }
    
    ResolveDependencies --> Finalize: Dependencies resolved
    state ResolveDependencies {
        [*] --> FindUnresolvedBlocks
        FindUnresolvedBlocks --> GatherContext
        GatherContext --> EnhanceDescriptions
        EnhanceDescriptions --> UpdateComponents
        UpdateComponents --> [*]
    }
    
    Finalize --> [*]: Analysis complete
    state Finalize {
        [*] --> GenerateDeepDescriptions
        GenerateDeepDescriptions --> UpdateMetadata
        UpdateMetadata --> [*]
    }
```

## Key State Additions Summary:

1. **Component-level tracking**:
   - `component_id`: Unique identifier (block_id:component_number)
   - `external_dependencies`: References to other blocks
   - `enhanced_description`: Updated after dependency resolution

2. **Block-level progress**:
   - `initial_pass_complete`: First analysis done
   - `dependency_resolution_complete`: Dependencies resolved
   - `is_fully_analyzed`: All analysis complete

3. **Global registries** for fast lookups:
   - `global_component_registry`: All components by ID
   - `function_registry`: Function name → component ID
   - `class_registry`: Class name → component ID
   - `import_registry`: Library → list of component IDs

4. **Progress tracking**:
   - `analysis_phase`: Current phase of analysis
   - `blocks_analyzed`, `dependencies_resolved`: Counters
   - Helper methods like `get_progress()` and `get_unresolved_blocks()`

5. **LangGraph specific**:
   - `messages`: Status updates for UI
   - `errors`: Error tracking
   - Proper state transitions between nodes

### Testing Component Extraction

In [304]:
# # Test 1: Basic Component Extraction
# print("=== TEST 1: Basic Component Extraction ===")

# # First, get the block-level analysis
# explanation = analyze_code('tiny_demo.ipynb')
# print(f"Found {len(explanation.blocks)} blocks\n")

# # Test extracting components from the first block (imports)
# if explanation.blocks:
#     first_block = explanation.blocks[0]
#     print(f"Testing Block: {first_block.name}")
#     print(f"Block Type: {first_block.id}")
#     print(f"Block Content Preview: {first_block.content[:100]}...\n")
    
#     # Extract components
#     components = analyze_block_components(
#         first_block.model_dump(),
#         explanation,
#         0
#     )
    
#     print(f"Found {len(components.components)} components:")
#     for comp in components.components:
#         print(f"\nComponent {comp.number}: {comp.name}")
#         print(f"  Type: {comp.type}")
#         print(f"  Library: {comp.library}")
#         print(f"  Description: {comp.description[:100]}...")
#         print(f"  Code: {comp.code[:50]}...")
#         if comp.calls:
#             print(f"  Calls: {comp.calls}")
#         if comp.parent:
#             print(f"  Parent: {comp.parent}")

In [305]:
# # Test 2: Test Component Relationships
# print("\n=== TEST 2: Component Relationships ===")

# # Test with a block that has functions/classes
# for i, block in enumerate(explanation.blocks):
#     if "def " in block.content or "class " in block.content:
#         print(f"\nTesting Block {block.id}: {block.name}")
        
#         components = analyze_block_components(
#             block.model_dump(),
#             explanation,
#             i
#         )
        
#         # Check for relationships
#         for comp in components.components:
#             if comp.calls or comp.called_by or comp.children:
#                 print(f"\nComponent {comp.number}: {comp.name}")
#                 if comp.calls:
#                     print(f"  Calls components: {comp.calls}")
#                 if comp.called_by:
#                     print(f"  Called by: {comp.called_by}")
#                 if comp.children:
#                     print(f"  Has children: {comp.children}")
#                 if comp.parent:
#                     print(f"  Parent component: {comp.parent}")
#         break

In [306]:
# # Test 3: Full Analysis with JSON Output
# print("\n=== TEST 3: Full Component Analysis JSON ===")

# # Analyze all blocks and collect results
# all_components = {}
# for i, block in enumerate(explanation.blocks):
#     components = analyze_block_components(
#         block.model_dump(),
#         explanation,
#         i
#     )
#     all_components[block.id] = {
#         "block_name": block.name,
#         "component_count": len(components.components),
#         "components": [comp.model_dump() for comp in components.components]
#     }

# # Pretty print one block's components as JSON
# if "3" in all_components:  # VectorStore class block
#     print(f"\nJSON output for Block 3 (VectorStore):")
#     print(json.dumps(all_components["3"], indent=2))

In [307]:
# # Test 4: Test Dependency Context
# print("\n=== TEST 4: Dependency Context ===")

# # Test the format_connected_blocks function
# if len(explanation.edges) > 0:
#     # Find a block that has dependencies
#     for block in explanation.blocks:
#         if block.dependencies:
#             print(f"\nBlock {block.id} depends on: {block.dependencies}")
            
#             # Format connected blocks context
#             connected_context = format_connected_blocks(
#                 explanation.edges,
#                 [b.model_dump() for b in explanation.blocks],
#                 block.id
#             )
            
#             print(f"\nConnected blocks context:")
#             print(connected_context)
#             break
# else:
#     print("No dependencies found in this simple example")
    
# # Summary
# print("\n=== SUMMARY ===")
# print(f"Total blocks analyzed: {len(explanation.blocks)}")
# print(f"Total components found: {sum(len(comps['components']) for comps in all_components.values())}")
# print(f"Blocks with dependencies: {len([b for b in explanation.blocks if b.dependencies])}")

### Testing Full LangGraph Workflow

In [308]:
# 🔨 STEP 3: RUN COMPONENT EXTRACTION AND DEPENDENCY RESOLUTION
print("\n🔍 STEP 3: COMPONENT ANALYSIS WITH LANGGRAPH")
print("="*60)

global final_state

# Create and run the LangGraph analysis pipeline
print("🔧 Creating LangGraph analysis pipeline...")
graph = create_analysis_graph()

# Initialize state with our target notebook
initial_state = WalkthroughState(
    notebook_path=NOTEBOOK_TO_ANALYZE,
    code_overview=""  # Will be populated by the graph
)

print("🚀 Running complete analysis pipeline...")
print("   📋 Stage 1: Block extraction and component analysis")
print("   🔗 Stage 2: Dependency resolution")
print("   📝 Stage 3: Enhanced descriptions")

# Run the complete graph
result = await graph.ainvoke(initial_state)
final_state = WalkthroughState(**result)

# Save complete final state
save_step("3_components", "final_state.json", final_state, "Complete component analysis state")

# Display results
print(f"\n📊 ANALYSIS RESULTS:")
progress = final_state.get_progress()
print(f"   ✅ Analysis phase: {progress['phase']}")
print(f"   ✅ Blocks analyzed: {progress['blocks_analyzed']}")
print(f"   ✅ Dependencies resolved: {progress['dependencies_resolved']}")
print(f"   📊 Total components: {final_state.total_components}")

# Save progress summary
save_step("3_components", "progress_summary.json", progress, "Analysis progress summary")

print(f"\n🏗️ BLOCK DETAILS:")
block_details = []
for block_id, block_state in final_state.block_states.items():
    print(f"   Block {block_id}: {block_state.block_name} ({block_state.block_type})")
    print(f"      Components: {len(block_state.component_states)}")
    print(f"      Complete: {block_state.is_fully_analyzed}")
    
    # Collect block details
    block_details.append({
        "block_id": block_id,
        "block_name": block_state.block_name,
        "block_type": block_state.block_type,
        "component_count": len(block_state.component_states),
        "is_complete": block_state.is_fully_analyzed,
        "has_dependencies": len(block_state.depends_on_blocks) > 0,
        "dependency_count": len(block_state.depends_on_blocks)
    })

save_step("3_components", "block_details.json", block_details, "Detailed block analysis results")

print(f"\n📚 GLOBAL REGISTRIES:")
registries = {
    "functions": len(final_state.function_registry),
    "classes": len(final_state.class_registry),
    "imports": len(final_state.import_registry)
}
print(f"   Functions: {registries['functions']}")
print(f"   Classes: {registries['classes']}")
print(f"   Imports: {registries['imports']}")

# Save registry information
save_step("3_components", "global_registries.json", {
    "function_registry": final_state.function_registry,
    "class_registry": final_state.class_registry,
    "import_registry": final_state.import_registry,
    "summary": registries
}, "Global component registries")

# Show example components from first block
first_block_id = list(final_state.block_states.keys())[0]
first_block = final_state.block_states[first_block_id]
print(f"\n🔍 COMPONENTS IN FIRST BLOCK ({first_block.block_name}):")

component_examples = []
for comp_id, comp_state in list(first_block.component_states.items())[:3]:
    print(f"   • {comp_state.component_name} ({comp_state.type})")
    print(f"     {comp_state.description[:100]}...")
    
    component_examples.append({
        "component_id": comp_id,
        "name": comp_state.component_name,
        "type": comp_state.type,
        "description": comp_state.description,
        "library": comp_state.library,
        "code_preview": comp_state.full_code[:200] + "..." if len(comp_state.full_code) > 200 else comp_state.full_code
    })

if len(first_block.component_states) > 3:
    print(f"   ... and {len(first_block.component_states) - 3} more components")

# Save component examples
save_step("3_components", "component_examples.json", component_examples, "Example components from first block")

# Save all components for each block
for block_id, block_state in final_state.block_states.items():
    block_components = []
    for comp_id, comp_state in block_state.component_states.items():
        block_components.append({
            "component_id": comp_id,
            "name": comp_state.component_name,
            "number": comp_state.component_number,
            "type": comp_state.type,
            "library": comp_state.library,
            "description": comp_state.description,
            "enhanced_description": comp_state.enhanced_description,
            "full_code": comp_state.full_code,
            "is_resolved": comp_state.is_resolved,
            "external_dependencies": comp_state.external_dependencies
        })
    
    save_step("3_components", f"block_{block_id}_components.json", block_components, f"All components in block {block_id}")


🔍 STEP 3: COMPONENT ANALYSIS WITH LANGGRAPH
🔧 Creating LangGraph analysis pipeline...
🚀 Running complete analysis pipeline...
   📋 Stage 1: Block extraction and component analysis
   🔗 Stage 2: Dependency resolution
   📝 Stage 3: Enhanced descriptions
Successfully separated 'multi_agent.ipynb' into:
  - Python file: multi_agent.py
  - Markdown file: multi_agent.md
Successfully separated 'multi_agent.ipynb' into:
  - Python file: multi_agent.py
  - Markdown file: multi_agent.md
💾 Saved final_state.json to step3_components/ - Complete component analysis state

📊 ANALYSIS RESULTS:
   ✅ Analysis phase: complete
   ✅ Blocks analyzed: 26/26
   ✅ Dependencies resolved: 26/0
   📊 Total components: 189
💾 Saved progress_summary.json to step3_components/ - Analysis progress summary

🏗️ BLOCK DETAILS:
   Block 1: Environment Setup (imports)
      Components: 4
      Complete: True
   Block 2: Document Loading (imports)
      Components: 4
      Complete: True
   Block 3: Text Processing (imports)

In [309]:
# # Examine a specific block with dependencies
# print("\n=== EXAMINING BLOCK WITH DEPENDENCIES ===")

# # Find the build_rag_graph block (should have dependencies)
# for block_id, block_state in final_state.block_states.items():
#     if "build_rag_graph" in block_state.block_name.lower() or block_state.block_type == "main":
#         print(f"\nBlock {block_id}: {block_state.block_name}")
#         print(f"Depends on blocks: {block_state.depends_on_blocks}")
        
#         print("\nComponents with external dependencies:")
#         for comp_id, comp_state in block_state.component_states.items():
#             if comp_state.external_dependencies:
#                 print(f"\n  Component: {comp_state.component_name}")
#                 print(f"  Original: {comp_state.description[:100]}...")
#                 if comp_state.enhanced_description:
#                     print(f"  Enhanced: {comp_state.enhanced_description[:100]}...")
#                 print(f"  External deps: {comp_state.external_dependencies}")
        
#         break

# # Check the dependency graph
# print("\n=== DEPENDENCY GRAPH ===")
# for source, targets in final_state.dependency_graph.items():
#     print(f"Block {source} -> {targets}")

In [310]:
# # Create a simple visualization of the analysis
# print("\n=== ANALYSIS VISUALIZATION ===")

# # Show execution order
# print("\nSuggested reading order:")
# for i, block_id in enumerate(final_state.execution_order):
#     block = final_state.block_states[block_id]
#     print(f"{i+1}. Block {block_id}: {block.block_name} ({block.block_type})")

# # Show component types distribution
# print("\nComponent types found:")
# component_types = {}
# for block_state in final_state.block_states.values():
#     for comp_state in block_state.component_states.values():
#         component_types[comp_state.type] = component_types.get(comp_state.type, 0) + 1

# for comp_type, count in sorted(component_types.items()):
#     print(f"  {comp_type}: {count}")

# # Export final state to JSON for inspection
# print("\n=== EXPORTING RESULTS ===")
# export_data = {
#     "overview": final_state.code_overview,
#     "stats": {
#         "total_blocks": final_state.total_blocks,
#         "total_components": final_state.total_components,
#         "blocks_with_dependencies": final_state.blocks_with_dependencies
#     },
#     "execution_order": final_state.execution_order,
#     "blocks": {}
# }

# for block_id, block_state in final_state.block_states.items():
#     export_data["blocks"][block_id] = {
#         "name": block_state.block_name,
#         "type": block_state.block_type,
#         "component_count": len(block_state.component_states),
#         "is_complete": block_state.is_fully_analyzed
#     }

# print("Export data structure created successfully!")
# print(f"Blocks exported: {len(export_data['blocks'])}")

In [311]:
# 🔨 STEP 4: RUN DEEP DESCRIPTIONS AND ITERATIVE IMPROVEMENT
print("\n📚 STEP 4: DEEP DESCRIPTIONS + ITERATIVE IMPROVEMENT")
print("="*60)

global improved_state

# Generate deep descriptions for all blocks
print("🔍 Generating comprehensive block descriptions...")
deep_descriptions = await generate_all_deep_descriptions(final_state)
print(f"✅ Generated {len(deep_descriptions)} deep descriptions")

# Save deep descriptions
save_step("4_deep_descriptions", "deep_descriptions.json", deep_descriptions, "Generated deep descriptions for all blocks")

# Show expansion metrics
expansion_metrics = {}
if deep_descriptions:
    expansions = []
    for block_id, block_state in final_state.block_states.items():
        if block_state.deep_description:
            expansion = len(block_state.deep_description) / len(block_state.initial_description)
            expansions.append(expansion)
            expansion_metrics[block_id] = {
                "block_name": block_state.block_name,
                "initial_length": len(block_state.initial_description),
                "deep_length": len(block_state.deep_description),
                "expansion_ratio": expansion
            }
    
    avg_expansion = sum(expansions) / len(expansions) if expansions else 0
    print(f"   📏 Average expansion: {avg_expansion:.1f}x more detailed")
    
    expansion_metrics["summary"] = {
        "average_expansion": avg_expansion,
        "total_blocks_expanded": len(expansions),
        "max_expansion": max(expansions) if expansions else 0,
        "min_expansion": min(expansions) if expansions else 0
    }

# Save expansion metrics
save_step("4_deep_descriptions", "expansion_metrics.json", expansion_metrics, "Metrics on description expansion")

# Run iterative improvement
print(f"\n🔄 Running iterative improvement with component updates...")
improved_state = await smart_iterative_improvement(
    final_state,
    max_iterations=2,
    improvement_threshold=10.0,
    update_components=True
)

# Save improved state
save_step("4_deep_descriptions", "improved_state.json", improved_state, "State after iterative improvement")

print(f"\n📊 IMPROVEMENT RESULTS:")
print(f"   ✅ Final analysis phase: {improved_state.analysis_phase}")
print(f"   🧩 Total components: {improved_state.total_components}")

# Create improvement summary
improvement_summary = {
    "final_phase": improved_state.analysis_phase,
    "total_components": improved_state.total_components,
    "blocks_processed": len(improved_state.block_states),
    "components_enhanced": 0,
    "blocks_enhanced": 0
}

# Count enhanced components and blocks
for block_id, block_state in improved_state.block_states.items():
    if block_state.deep_description:
        improvement_summary["blocks_enhanced"] += 1
    
    for comp_state in block_state.component_states.values():
        if comp_state.enhanced_description:
            improvement_summary["components_enhanced"] += 1

save_step("4_deep_descriptions", "improvement_summary.json", improvement_summary, "Summary of improvement process")

# Show example of before/after for one block
example_block_id = list(improved_state.block_states.keys())[0]
example_block = improved_state.block_states[example_block_id]
print(f"\n📄 EXAMPLE DEEP DESCRIPTION ({example_block.block_name}):")
print("─" * 50)
print(f"INITIAL: {example_block.initial_description}")
deep_preview = example_block.deep_description[:300] + "..." if len(example_block.deep_description) > 300 else example_block.deep_description
print(f"\nDEEP: {deep_preview}")

# Save example comparison
example_comparison = {
    "block_id": example_block_id,
    "block_name": example_block.block_name,
    "initial_description": example_block.initial_description,
    "deep_description": example_block.deep_description,
    "improvement_ratio": len(example_block.deep_description) / len(example_block.initial_description) if example_block.initial_description else 0
}
save_step("4_deep_descriptions", "example_comparison.json", example_comparison, "Before/after comparison for example block")

# Show component enhancement example
example_components = list(example_block.component_states.values())[:2]
print(f"\n🔍 COMPONENT ENHANCEMENT EXAMPLES:")
component_enhancements = []
for comp in example_components:
    print(f"   • {comp.component_name} ({comp.type}):")
    print(f"     Initial: {comp.description[:80]}...")
    if comp.enhanced_description:
        print(f"     Enhanced: {comp.enhanced_description[:80]}...")
        status = "enhanced"
    else:
        print(f"     Enhanced: (no enhancement needed)")
        status = "no_enhancement_needed"
    
    component_enhancements.append({
        "component_name": comp.component_name,
        "type": comp.type,
        "initial_description": comp.description,
        "enhanced_description": comp.enhanced_description,
        "status": status
    })

save_step("4_deep_descriptions", "component_enhancements.json", component_enhancements, "Examples of component enhancements")

# Save all enhanced descriptions for each block
for block_id, block_state in improved_state.block_states.items():
    block_enhancement_data = {
        "block_id": block_id,
        "block_name": block_state.block_name,
        "initial_description": block_state.initial_description,
        "deep_description": block_state.deep_description,
        "enhanced_components": []
    }
    
    for comp_state in block_state.component_states.values():
        if comp_state.enhanced_description:
            block_enhancement_data["enhanced_components"].append({
                "component_name": comp_state.component_name,
                "type": comp_state.type,
                "initial": comp_state.description,
                "enhanced": comp_state.enhanced_description
            })
    
    save_step("4_deep_descriptions", f"block_{block_id}_enhancements.json", block_enhancement_data, f"All enhancements for block {block_id}")

print(f"\n✅ Deep analysis and improvement complete!")


📚 STEP 4: DEEP DESCRIPTIONS + ITERATIVE IMPROVEMENT
🔍 Generating comprehensive block descriptions...
Generating deep descriptions for 0 blocks...
✅ Generated 0 deep descriptions
💾 Saved deep_descriptions.json to step4_deep_descriptions/ - Generated deep descriptions for all blocks
💾 Saved expansion_metrics.json to step4_deep_descriptions/ - Metrics on description expansion

🔄 Running iterative improvement with component updates...

=== STARTING ITERATIVE IMPROVEMENT ===

--- Iteration 1 ---
Generating comprehensive overview...
Assessing improvement potential...
Improvement score: 45.0%
Rationale: The deep analysis revealed significant architectural insights, particularly regarding the modular design and the roles of various agents. However, there are still areas where additional iteration could clarify dependencies and enhance the system's capabilities, especially in integrating external data sources and optimizing agent interactions.

Update plan:
  - Update overview: True
  - Blocks

In [312]:
# # Show full deep description for one example block
# print("\n=== FULL DEEP DESCRIPTION EXAMPLE ===\n")

# # Let's show the VectorStore class block as an example
# example_block_id = "3"  # Usually the class definition block
# if example_block_id in final_state.block_states:
#     example_block = final_state.block_states[example_block_id]
    
#     print(f"Block: {example_block.block_name}")
#     print("="*80)
#     print("\nINITIAL DESCRIPTION:")
#     print("-"*40)
#     print(example_block.initial_description)
    
#     print("\n\nDEEP DESCRIPTION:")
#     print("-"*40)
#     print(example_block.deep_description)
    
#     print("\n\nCOMPONENT DETAILS:")
#     print("-"*40)
#     for comp_id, comp_state in example_block.component_states.items():
#         print(f"\n{comp_state.component_name} ({comp_state.type}):")
#         print(f"  Initial: {comp_state.description}")
#         if comp_state.enhanced_description:
#             print(f"  Enhanced: {comp_state.enhanced_description}")
# else:
#     # Try to find any class block
#     for block_id, block_state in final_state.block_states.items():
#         if block_state.block_type == "classes":
#             example_block = block_state
#             print(f"Block: {example_block.block_name}")
#             print("="*80)
#             print("\nDEEP DESCRIPTION:")
#             print(example_block.deep_description)
#             break

In [313]:
# # Test Iterative Improvement System with Component Updates
# print("=== TESTING ITERATIVE IMPROVEMENT SYSTEM WITH COMPONENT UPDATES ===\n")

# # Ensure we have a state with deep descriptions
# if 'final_state' not in globals() or not all(bs.deep_description for bs in final_state.block_states.values()):
#     print("Running full analysis first...")
#     final_state = await test_langgraph_workflow()
#     # Clear and regenerate deep descriptions
#     for block_state in final_state.block_states.values():
#         block_state.deep_description = None
#     await generate_all_deep_descriptions(final_state)

# # Store original state for comparison
# original_overview = final_state.code_overview
# original_deep_descs = {
#     block_id: block_state.deep_description 
#     for block_id, block_state in final_state.block_states.items()
# }

# # Store original component descriptions
# original_comp_descs = {}
# for block_id, block_state in final_state.block_states.items():
#     for comp_id, comp_state in block_state.component_states.items():
#         original_comp_descs[comp_id] = {
#             'initial': comp_state.description,
#             'enhanced': comp_state.enhanced_description
#         }

# # Run iterative improvement with component updates
# improved_state = await smart_iterative_improvement(
#     final_state,
#     max_iterations=2,
#     improvement_threshold=10.0,
#     update_components=True  # Enable component updates
# )

# # Show results
# print("\n=== IMPROVEMENT RESULTS ===")
# print("\nORIGINAL OVERVIEW:")
# print("-" * 40)
# print(original_overview[:300] + "...")

# print("\n\nIMPROVED OVERVIEW:")
# print("-" * 40)
# print(improved_state.code_overview[:300] + "...")

# # Check which blocks were updated
# updated_blocks = []
# for block_id, block_state in improved_state.block_states.items():
#     if block_id in original_deep_descs:
#         if block_state.deep_description != original_deep_descs[block_id]:
#             updated_blocks.append(block_id)

# print(f"\n\nBLOCKS UPDATED: {len(updated_blocks)}")

# # Check which components were updated
# updated_components = []
# for block_id, block_state in improved_state.block_states.items():
#     for comp_id, comp_state in block_state.component_states.items():
#         if comp_id in original_comp_descs:
#             orig = original_comp_descs[comp_id]
#             if comp_state.enhanced_description and comp_state.enhanced_description != orig['enhanced']:
#                 updated_components.append((comp_id, comp_state))

# print(f"\nCOMPONENTS UPDATED: {len(updated_components)}")

# # Show component update examples
# if updated_components:
#     print("\n=== COMPONENT UPDATE EXAMPLES ===")
#     for i, (comp_id, comp_state) in enumerate(updated_components[:3]):  # Show first 3
#         print(f"\n--- Component: {comp_state.component_name} ({comp_state.type}) ---")
#         print(f"Original: {comp_state.description}")
#         print(f"Enhanced: {comp_state.enhanced_description}")
        
# # Summary statistics
# print(f"\n=== SUMMARY ===")
# print(f"Overview expanded: {((len(improved_state.code_overview) - len(original_overview)) / len(original_overview) * 100):.1f}%")
# print(f"Blocks updated: {len(updated_blocks)}/{len(improved_state.block_states)}")
# print(f"Components enhanced: {len(updated_components)}/{sum(len(bs.component_states) for bs in improved_state.block_states.values())}")

# 📖 STEP 5: Educational Walkthrough Generation

This final step creates an interactive educational walkthrough with:
- **Teaching Plan**: Logical ordering of concepts for learning
- **Component Linking**: Clickable links in special markdown format for VS Code integration
- **100% Coverage**: Ensures all components are mentioned and linked
- **Advanced Post-processing**: Validates and enhances component links using regex patterns

The walkthrough uses the format `[[component:block_id:component_number:safe_name|display_text]]` for VS Code extension compatibility.

In [314]:
# Walkthrough Generation System
from typing import List, Dict, Optional, Set, Tuple
from pydantic import BaseModel, Field
from dataclasses import dataclass
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langsmith import traceable
import json
import asyncio
import re

# Data models for walkthrough
@dataclass
class ComponentLink:
    """Represents a clickable component reference"""
    block_id: str
    component_number: int
    component_name: str
    display_text: str
    
    def to_markdown(self) -> str:
        """Convert to special markdown format for VS Code extension"""
        # Ensure component name has no spaces for better compatibility
        safe_name = self.component_name.replace(" ", "_")
        return f"[[component:{self.block_id}:{self.component_number}:{safe_name}|{self.display_text}]]"

class WalkthroughPlan(BaseModel):
    """Teaching plan for the entire codebase"""
    introduction: str = Field(description="Brief introduction explaining the codebase purpose")
    block_plans: Dict[str, List[str]] = Field(default_factory=dict, description="Teaching points for each block")
    suggested_order: List[str] = Field(default_factory=list, description="Suggested block reading order")

class WalkthroughSection(BaseModel):
    """A single section of the walkthrough (one per block)"""
    block_id: str
    block_name: str
    content: str  # Contains component links in markdown
    components_referenced: List[Dict] = Field(default_factory=list)  # Track which components were mentioned
    teaching_points_covered: List[str] = Field(default_factory=list)

class Walkthrough(BaseModel):
    """Complete walkthrough document"""
    introduction: str
    sections: List[WalkthroughSection]
    total_components: int
    components_covered: int
    coverage_report: Dict[str, List[str]] = Field(default_factory=dict)  # block_id -> missing components

# Dynamic component name mapping function
def build_component_name_map(walkthrough_state: WalkthroughState) -> Dict[str, str]:
    """Build a dynamic component name mapping from the walkthrough state"""
    component_map = {}
    
    for block_id, block_state in walkthrough_state.block_states.items():
        for comp_id, comp_state in block_state.component_states.items():
            key = f"{block_id}:{comp_state.component_number}"
            # Make component names safe for linking
            safe_name = comp_state.component_name.replace(" ", "_").replace("-", "_")
            component_map[key] = safe_name
    
    return component_map

# Cross-reference mapping function 
def build_cross_reference_map(walkthrough_state: WalkthroughState) -> Dict[str, List[str]]:
    """Build a map of component names to all their occurrences across blocks"""
    cross_refs = {}
    
    for block_id, block_state in walkthrough_state.block_states.items():
        for comp_state in block_state.component_states.values():
            comp_name = comp_state.component_name
            if comp_name not in cross_refs:
                cross_refs[comp_name] = []
            
            safe_name = comp_state.component_name.replace(" ", "_").replace("-", "_")
            link = f"[[component:{block_id}:{comp_state.component_number}:{safe_name}|{comp_name}]]"
            cross_refs[comp_name].append(link)
    
    return cross_refs

# Global component name map (will be populated dynamically)
COMPONENT_NAME_MAP = {}

# Prompts for walkthrough generation
WALKTHROUGH_PLANNER_PROMPT = """You are creating a teaching plan for explaining a codebase to someone learning it.

## Codebase Overview:
{comprehensive_overview}

## Code Blocks:
{block_summaries}

## Execution Order:
{execution_order}

## Task:
Create a teaching plan that:
1. Writes a brief introduction (2-3 sentences) explaining what this code does and why it's useful
2. For each block, creates 3-5 teaching points that:
   - Follow a logical learning progression
   - Build on concepts from previous blocks
   - Ensure all major functionality is covered
   - Consider the dependencies between blocks

## Output Format:
Return a JSON object with:
- "introduction": Brief, engaging introduction
- "block_plans": Object mapping block_id to array of teaching points
- "suggested_order": Array of block_ids in recommended learning order

Example:
{{
  "introduction": "This pipeline processes text documents...",
  "block_plans": {{
    "1": ["Understanding the required libraries", "Why we need each import"],
    "2": ["Text preprocessing fundamentals", "How cleaning improves results"],
    ...
  }},
  "suggested_order": ["1", "2", "3", "4", "5"]
}}
"""

# Enhanced Block Walkthrough Prompt with mandatory linking and cross-references
BLOCK_WALKTHROUGH_PROMPT = """You are writing an educational walkthrough for a specific code block.

## BLOCK INFORMATION:
Block {block_index}: {block_name}
Overview: {overview}
Deep Description: {deep_description}

## COMPONENT DESCRIPTIONS:
{component_descriptions}

## COMPONENT LINK MAPPING:
Below are the EXACT links you must use when mentioning any component by name. When you need to mention a component, find it in this map and use the provided link format:

{component_link_map}

## CROSS-REFERENCE MAPPING:
Some components appear in multiple blocks. Here are all occurrences:
{cross_reference_map}

## INSTRUCTIONS:
Create a comprehensive educational walkthrough that:
1. Explains the block's purpose and architecture
2. Integrates ALL component descriptions naturally
3. Uses the EXACT component links from the mapping above
4. When referring to components from other blocks, use their specific block's link
5. Maintain narrative flow while being technically accurate

IMPORTANT: 
- Use the EXACT link format from the component link mapping
- Every component name mentioned MUST use its corresponding link
- Do not create new links - only use the ones provided in the mapping
- When a component appears in multiple blocks, choose the most contextually appropriate link
"""

# Helper function to get safe component name
def get_safe_component_name(block_id: str, comp_num: int, original_name: str) -> str:
    """Get the safe component name from mapping or create one"""
    key = f"{block_id}:{comp_num}"
    if key in COMPONENT_NAME_MAP:
        return COMPONENT_NAME_MAP[key]
    # Fallback: replace spaces and hyphens with underscores
    return original_name.replace(" ", "_").replace("-", "_")

# Core walkthrough generation functions
@traceable
async def create_walkthrough_plan(walkthrough_state: WalkthroughState) -> WalkthroughPlan:
    """Create a teaching plan for the walkthrough"""
    
    # Prepare block summaries
    block_summaries = []
    for block_id in walkthrough_state.execution_order:
        if block_id in walkthrough_state.block_states:
            block = walkthrough_state.block_states[block_id]
            summary = f"Block {block_id} ({block.block_name}): {block.initial_description}"
            block_summaries.append(summary)
    
    # Format the prompt
    formatted_prompt = WALKTHROUGH_PLANNER_PROMPT.format(
        comprehensive_overview=walkthrough_state.code_overview,
        block_summaries="\n".join(block_summaries),
        execution_order=", ".join(walkthrough_state.execution_order)
    )
    
    # Get structured response - use function_calling method for compatibility
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)  # Higher temp for creativity
    structured_llm = llm.with_structured_output(WalkthroughPlan, method="function_calling")
    
    messages = [
        SystemMessage(content=formatted_prompt),
        HumanMessage(content="Create the teaching plan for this codebase.")
    ]
    
    plan = await structured_llm.ainvoke(messages)
    
    # Ensure all blocks have plans
    for block_id in walkthrough_state.block_states.keys():
        if block_id not in plan.block_plans:
            plan.block_plans[block_id] = ["Explain this block's purpose", "Cover main components"]
    
    return plan

@traceable
async def generate_block_walkthrough(block_id: str, walkthrough_state: WalkthroughState) -> str:
    """Generate walkthrough content for a single block with component links"""
    block_state = walkthrough_state.block_states[block_id]
    
    # Build component descriptions
    component_descriptions = []
    for comp_state in block_state.component_states.values():
        # Use enhanced_description if available, otherwise use description
        final_desc = comp_state.enhanced_description or comp_state.description
        desc = f"{comp_state.component_number}. **{comp_state.component_name}** ({comp_state.type}): {final_desc}"
        component_descriptions.append(desc)
    
    # Build component link map with variations
    component_link_map = []
    for comp_state in block_state.component_states.values():
        safe_name = comp_state.component_name.replace(" ", "_").replace("-", "_")
        link = f"[[component:{block_id}:{comp_state.component_number}:{safe_name}|{{display_text}}]]"
        
        # Add variations of the component name
        variations = [
            comp_state.component_name,
            f"{comp_state.component_name} function",
            f"{comp_state.component_name} method",
            f"{comp_state.component_name} class",
            f"{comp_state.component_name} variable",
            f"{comp_state.component_name}()",
            f"`{comp_state.component_name}`"
        ]
        
        for variation in variations:
            component_link_map.append(f'"{variation}": {link.format(display_text=variation)}')
    
    # Get cross-references
    cross_refs = build_cross_reference_map(walkthrough_state)
    cross_ref_text = json.dumps(cross_refs, indent=2)
    
    # Build the prompt with correct attributes
    prompt = BLOCK_WALKTHROUGH_PROMPT.format(
        block_index=block_id,
        block_name=block_state.block_name,
        overview=block_state.initial_description,
        deep_description=block_state.deep_description or block_state.initial_description,
        component_descriptions="\n".join(component_descriptions),
        component_link_map="{\n" + ",\n".join(component_link_map) + "\n}",
        cross_reference_map=cross_ref_text
    )
    
    chain = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)
    response = await chain.ainvoke(prompt)
    
    return response.content

@traceable
async def generate_complete_walkthrough(walkthrough_state: WalkthroughState) -> Walkthrough:
    """Generate the complete walkthrough with all sections"""
    
    print("\n=== GENERATING ENHANCED WALKTHROUGH ===")
    
    # Build dynamic component name map
    global COMPONENT_NAME_MAP
    COMPONENT_NAME_MAP = build_component_name_map(walkthrough_state)
    print(f"  ✓ Built component name map with {len(COMPONENT_NAME_MAP)} components")
    
    # 1. Create teaching plan
    print("Creating teaching plan...")
    plan = await create_walkthrough_plan(walkthrough_state)
    print(f"  ✓ Created plan with {len(plan.block_plans)} blocks")
    
    # 2. Generate sections with enhanced linking
    print("\nGenerating block walkthroughs with mandatory component linking...")
    section_tasks = []
    
    for block_id in plan.suggested_order:
        if block_id in walkthrough_state.block_states:
            task = generate_block_walkthrough(
                block_id=block_id,
                walkthrough_state=walkthrough_state
            )
            section_tasks.append(task)
    
    walkthrough_contents = await asyncio.gather(*section_tasks)
    
    # Create WalkthroughSection objects
    sections = []
    for i, (block_id, content) in enumerate(zip(plan.suggested_order, walkthrough_contents)):
        if block_id in walkthrough_state.block_states:
            block_state = walkthrough_state.block_states[block_id]
            
            # Extract component references
            component_pattern = r'\[\[component:(\w+):(\d+):([^|]+)\|([^\]]+)\]\]'
            matches = re.findall(component_pattern, content)
            
            components_referenced = []
            for match in matches:
                components_referenced.append({
                    "block_id": match[0],
                    "component_number": int(match[1]),
                    "component_name": match[2],
                    "display_text": match[3]
                })
            
            section = WalkthroughSection(
                block_id=block_id,
                block_name=block_state.block_name,
                content=content,
                components_referenced=components_referenced,
                teaching_points_covered=plan.block_plans.get(block_id, [])
            )
            sections.append(section)
    
    print(f"  ✓ Generated {len(sections)} block walkthroughs")
    
    # 3. Create walkthrough
    walkthrough = Walkthrough(
        introduction=plan.introduction,
        sections=sections,
        total_components=sum(len(block.component_states) for block in walkthrough_state.block_states.values()),
        components_covered=len(set(comp_ref['block_id'] + ":" + str(comp_ref['component_number']) 
                                    for section in sections for comp_ref in section.components_referenced)),
        coverage_report={}
    )
    
    print("\n=== ENHANCED WALKTHROUGH COMPLETE ===")
    
    return walkthrough

# Helper function to render walkthrough as markdown
def render_walkthrough_markdown(walkthrough: Walkthrough) -> str:
    """Render the walkthrough as a complete markdown document"""
    
    md_parts = []
    
    # Title and introduction
    md_parts.append("# Code Walkthrough\n")
    md_parts.append(walkthrough.introduction)
    md_parts.append("\n---\n")
    
    # Sections
    for i, section in enumerate(walkthrough.sections):
        md_parts.append(f"\n## {i+1}. {section.block_name}\n")
        md_parts.append(section.content)
        md_parts.append("\n")
    
    return "".join(md_parts)

# ROBUST DYNAMIC PREVIEW GENERATION FUNCTION - WORKS FOR ANY NOTEBOOK
def create_preview_walkthrough(walkthrough: Walkthrough) -> str:
    """Convert custom component syntax to standard markdown links for preview - Universal Solution"""
    
    # Get the original markdown
    markdown_content = render_walkthrough_markdown(walkthrough)
    
    print(f"🔄 Starting preview conversion for {len(markdown_content)} characters")
    
    # COMPREHENSIVE PATTERN SET - handles all possible component link variations
    patterns = [
        # Pattern 1: Nested with backticks: [**`[[component:1:1:name|display**](#link)`]]
        (r'\[\*\*`\[\[component:([^:\]]+):([^:\]]+):([^|\]]+)\|([^\]]+)\*\*\]\([^)]+\)`\]\]', 
         lambda m: f'[**{m.group(4)}**](#component-{m.group(1)}-{m.group(2)}-{m.group(3).replace("_", "-").lower()})', 
         "Nested backtick pattern"),
        
        # Pattern 2: Nested double brackets: [**[[component:1:1:name|display**](#link)]]
        (r'\[\*\*\[\[component:([^:\]]+):([^:\]]+):([^|\]]+)\|([^\]]+)\*\*\]\([^)]+\)\]\]', 
         lambda m: f'[**{m.group(4)}**](#component-{m.group(1)}-{m.group(2)}-{m.group(3).replace("_", "-").lower()})', 
         "Nested double bracket pattern"),
        
        # Pattern 3: Standard component syntax: [[component:1:2:name|display]]
        (r'\[\[component:([^:\]]+):([^:\]]+):([^|\]]+)\|([^\]]+)\]\]', 
         lambda m: f'[**{m.group(4)}**](#component-{m.group(1)}-{m.group(2)}-{m.group(3).replace("_", "-").lower()})', 
         "Standard component pattern"),
        
        # Pattern 4: Empty component name: [[component:1:2:|display]]
        (r'\[\[component:([^:\]]+):([^:\]]+):\|([^\]]+)\]\]', 
         lambda m: f'[**{m.group(3)}**](#component-{m.group(1)}-{m.group(2)})', 
         "Empty component name pattern"),
        
        # Pattern 5: Missing component section: [[component:1:2|display]]
        (r'\[\[component:([^:\]]+):([^:\]]+)\|([^\]]+)\]\]', 
         lambda m: f'[**{m.group(3)}**](#component-{m.group(1)}-{m.group(2)})', 
         "Missing component section pattern"),
        
        # Pattern 6: Catch-all for any remaining [[component:...]] patterns
        (r'\[\[component:([^\]]+)\]\]', 
         lambda m: f'[**{m.group(1).split("|")[-1] if "|" in m.group(1) else m.group(1)}**](#component-link)', 
         "Catch-all component pattern"),
    ]
    
    result = markdown_content
    total_conversions = 0
    
    # Apply patterns in multiple passes to handle complex nested cases
    for pass_num in range(4):  # Up to 4 passes for maximum coverage
        pass_conversions = 0
        
        for i, (pattern, replacement_func, description) in enumerate(patterns):
            matches = list(re.finditer(pattern, result))
            if matches:
                print(f"  Pass {pass_num + 1}, Pattern {i + 1} ({description}): Found {len(matches)} matches")
                new_result = re.sub(pattern, replacement_func, result)
                if new_result != result:
                    pass_conversions += len(matches)
                    result = new_result
        
        total_conversions += pass_conversions
        print(f"  Pass {pass_num + 1} complete: {pass_conversions} conversions")
        
        # If no changes in this pass, we're done
        if pass_conversions == 0:
            print(f"  No more conversions needed after pass {pass_num + 1}")
            break
    
    # CLEANUP PHASE - Fix any remaining malformed patterns
    print("🧹 Cleaning up malformed patterns...")
    cleanup_patterns = [
        # Fix: `[**TEXT**](#link)` -> **TEXT**
        (r'`\[\*\*([^*]+)\*\*\]\([^)]+\)', r'**\1**', "Backtick cleanup"),
        
        # Fix: [**`text`**](#link)] -> [**text**](#component-link)
        (r'\[\*\*`([^`]+)`\*\*\]\([^)]+\)\]', r'[**\1**](#component-link)', "Trailing bracket cleanup"),
        
        # Fix: **text**` -> **text**
        (r'\*\*([^*]+)\*\*`', r'**\1**', "Trailing backtick cleanup"),
        
        # Fix: `**text** -> **text**
        (r'`\*\*([^*]+)\*\*', r'**\1**', "Leading backtick cleanup"),
    ]
    
    cleanup_conversions = 0
    for pattern, replacement, description in cleanup_patterns:
        matches = len(re.findall(pattern, result))
        if matches > 0:
            result = re.sub(pattern, replacement, result)
            cleanup_conversions += matches
            print(f"  {description}: Fixed {matches} patterns")
    
    # Add header explaining the preview
    preview_header = """# Code Walkthrough (Preview Mode)

> **Note**: This is a preview version where component links are displayed as clickable markdown links. 
> In the VS Code extension, these would navigate directly to the source code.

---

"""
    
    final_result = preview_header + result
    
    print(f"✅ Preview conversion complete:")
    print(f"  📊 Total component link conversions: {total_conversions}")
    print(f"  🧹 Cleanup fixes applied: {cleanup_conversions}")
    print(f"  📏 Final content length: {len(final_result):,} characters")
    print(f"  🎯 Ready for markdown preview!")
    
    return final_result

# Function to save both versions automatically
def save_walkthrough_with_preview(walkthrough: Walkthrough, base_filename: str):
    """Save both original and preview versions of the walkthrough"""
    
    # Save original version (for VS Code extension)
    original_content = render_walkthrough_markdown(walkthrough)
    save_step("5_walkthrough", f"{base_filename}.md", original_content, "Original walkthrough with component syntax")
    
    # Save preview version (for markdown viewers) - now with robust conversion
    preview_content = create_preview_walkthrough(walkthrough)
    save_step("5_walkthrough", f"{base_filename}_preview.md", preview_content, "Preview walkthrough with standard markdown links")
    
    print(f"💾 Saved both versions:")
    print(f"  📄 {base_filename}.md - Original (for VS Code extension)")
    print(f"  👁️ {base_filename}_preview.md - Preview (for markdown viewers)")
    
    return original_content, preview_content

# Example VS Code extension parsing function (for reference)
def parse_component_links(text: str) -> List[Dict]:
    """Parse component links from walkthrough text"""
    pattern = r'\[\[component:(\w+):(\d+):([^|]+)\|([^\]]+)\]\]'
    matches = re.findall(pattern, text)
    
    links = []
    for match in matches:
        links.append({
            "block_id": match[0],
            "component_number": int(match[1]),
            "component_name": match[2],
            "display_text": match[3],
            "full_link": f"[[component:{match[0]}:{match[1]}:{match[2]}|{match[3]}]]"
        })
    
    return links

In [315]:
# Advanced Post-processing and Validation Functions
import re
from typing import Dict, List, Tuple, Set, Optional
from dataclasses import dataclass
from collections import defaultdict

@dataclass
class ComponentVariation:
    """Track different variations of component names"""
    canonical_name: str
    block_id: str
    component_number: int
    variations: Set[str]
    priority: int  # Higher priority = more likely to be the correct match

class ComponentLookupDictionary:
    """Comprehensive lookup dictionary for component names and variations"""
    
    def __init__(self, walkthrough_state: WalkthroughState):
        self.component_variations: Dict[str, ComponentVariation] = {}
        self.name_to_id_map: Dict[str, str] = {}  # name -> component_id
        self.fuzzy_matches: Dict[str, List[str]] = defaultdict(list)
        self._build_dictionary(walkthrough_state)
    
    def _build_dictionary(self, walkthrough_state: WalkthroughState):
        """Build comprehensive lookup dictionary from state"""
        
        for block_id, block_state in walkthrough_state.block_states.items():
            for comp_id, comp_state in block_state.component_states.items():
                canonical_name = comp_state.component_name
                comp_key = f"{block_id}:{comp_state.component_number}"
                
                # Generate variations for this component
                variations = self._generate_name_variations(canonical_name, comp_state.type)
                
                # Determine priority based on component type and context
                priority = self._calculate_priority(comp_state.type, canonical_name, comp_state.description)
                
                # Store component variation
                comp_variation = ComponentVariation(
                    canonical_name=canonical_name,
                    block_id=block_id,
                    component_number=comp_state.component_number,
                    variations=variations,
                    priority=priority
                )
                
                self.component_variations[comp_key] = comp_variation
                
                # Build reverse lookup maps
                for variation in variations:
                    self.name_to_id_map[variation.lower()] = comp_key
                    # Add to fuzzy matches for partial matching
                    self.fuzzy_matches[variation.lower()[:5]].append(comp_key)
    
    def _generate_name_variations(self, name: str, comp_type: str) -> Set[str]:
        """Generate all possible variations of a component name"""
        variations = {name}
        
        # Basic transformations
        variations.add(name.lower())
        variations.add(name.upper())
        variations.add(name.replace(" ", "_"))
        variations.add(name.replace("_", " "))
        variations.add(name.replace("-", "_"))
        variations.add(name.replace("_", "-"))
        
        # Remove common prefixes/suffixes
        clean_name = name.replace("Import_", "").replace("_library", "").replace("_function", "").replace("_class", "")
        variations.add(clean_name)
        
        # Type-specific variations
        if comp_type == "function":
            variations.add(f"{name}_function")
            variations.add(f"{name}()")
            variations.add(f"function {name}")
            
        elif comp_type == "class":
            variations.add(f"{name}_class")
            variations.add(f"class {name}")
            variations.add(f"{name} class")
            
        elif comp_type == "import":
            # Handle import variations
            if "import" in name.lower():
                variations.add(name.replace("Import_", "").replace("_library", ""))
            variations.add(f"import {clean_name}")
            variations.add(f"{clean_name} import")
            
        elif comp_type == "variable":
            variations.add(f"{name}_variable")
            variations.add(f"variable {name}")
            
        elif comp_type == "method":
            variations.add(f"{name}_method")
            variations.add(f"{name}()")
            variations.add(f"method {name}")
        
        # Common programming variations
        variations.add(name.strip())
        variations.add(f"`{name}`")
        variations.add(f'"{name}"')
        variations.add(f"'{name}'")
        
        return variations
    
    def _calculate_priority(self, comp_type: str, name: str, description: str) -> int:
        """Calculate priority for component matching"""
        base_priority = {
            "class": 100,
            "function": 90,
            "method": 80,
            "import": 70,
            "variable": 60,
            "expression": 50,
            "decorator": 40
        }.get(comp_type, 30)
        
        # Boost priority for key components
        if any(keyword in description.lower() for keyword in ["main", "core", "primary", "key"]):
            base_priority += 20
        
        # Boost for longer, more specific names
        if len(name) > 10:
            base_priority += 10
        
        return base_priority
    
    def find_component_id(self, text: str, context: str = "") -> Optional[str]:
        """Find component ID for a given text mention"""
        
        # Direct exact match
        if text.lower() in self.name_to_id_map:
            return self.name_to_id_map[text.lower()]
        
        # Fuzzy matching
        candidates = []
        for variation, comp_ids in self.fuzzy_matches.items():
            if variation in text.lower() or text.lower() in variation:
                for comp_id in comp_ids:
                    comp_var = self.component_variations[comp_id]
                    candidates.append((comp_id, comp_var.priority))
        
        # Return highest priority candidate
        if candidates:
            candidates.sort(key=lambda x: x[1], reverse=True)
            return candidates[0][0]
        
        return None
    
    def get_all_variations(self, component_id: str) -> Set[str]:
        """Get all variations for a component ID"""
        if component_id in self.component_variations:
            return self.component_variations[component_id].variations
        return set()

class AdvancedComponentLinker:
    """Advanced component linking with regex-based search and replace"""
    
    def __init__(self, lookup_dict: ComponentLookupDictionary):
        self.lookup_dict = lookup_dict
        self.link_patterns = self._build_link_patterns()
    
    def _build_link_patterns(self) -> List[Tuple[str, str, int]]:
        """Build regex patterns for component linking"""
        patterns = []
        
        for comp_id, comp_var in self.lookup_dict.component_variations.items():
            block_id = comp_var.block_id
            comp_num = comp_var.component_number
            safe_name = get_safe_component_name(block_id, comp_num, comp_var.canonical_name)
            
            for variation in comp_var.variations:
                # Skip very short variations to avoid false positives
                if len(variation) < 3:
                    continue
                
                # Build pattern with word boundaries and context awareness
                patterns.extend([
                    # Function calls
                    (rf'\b{re.escape(variation)}\s*\(', 
                     f'[[component:{block_id}:{comp_num}:{safe_name}|{variation}]](', 
                     comp_var.priority + 10),
                    
                    # Backtick mentions
                    (rf'`{re.escape(variation)}`', 
                     f'[[component:{block_id}:{comp_num}:{safe_name}|{variation}]]', 
                     comp_var.priority + 5),
                    
                    # Quoted mentions
                    (rf'"{re.escape(variation)}"', 
                     f'[[component:{block_id}:{comp_num}:{safe_name}|{variation}]]', 
                     comp_var.priority + 3),
                    
                    # Word boundary matches
                    (rf'\b{re.escape(variation)}\b(?!\s*[\(\)\[\]|])', 
                     f'[[component:{block_id}:{comp_num}:{safe_name}|{variation}]]', 
                     comp_var.priority),
                ])
        
        # Sort by priority (highest first)
        patterns.sort(key=lambda x: x[2], reverse=True)
        return patterns
    
    def apply_iterative_replacement(self, content: str, max_iterations: int = 3) -> Tuple[str, int]:
        """Apply iterative replacement with priority rules"""
        
        modified_content = content
        total_replacements = 0
        
        for iteration in range(max_iterations):
            iteration_replacements = 0
            
            # Track what we've already linked to avoid double-linking
            existing_links = set(re.findall(r'\[\[component:(\w+):(\d+):([^|]+)\|([^\]]+)\]\]', modified_content))
            
            for pattern, replacement, priority in self.link_patterns:
                # Skip if this would create nested links
                if '[[component:' in replacement and '[[component:' in modified_content:
                    # Check if replacement would create nested structure
                    if self._would_create_nested_links(modified_content, pattern):
                        continue
                
                # Apply replacement
                new_content = re.sub(pattern, replacement, modified_content, count=1, flags=re.IGNORECASE)
                
                if new_content != modified_content:
                    modified_content = new_content
                    iteration_replacements += 1
                    total_replacements += 1
                    
                    # Limit replacements per iteration to avoid runaway changes
                    if iteration_replacements >= 10:
                        break
            
            # Stop if no changes made this iteration
            if iteration_replacements == 0:
                break
        
        return modified_content, total_replacements
    
    def _would_create_nested_links(self, content: str, pattern: str) -> bool:
        """Check if applying pattern would create nested links"""
        # Simple check for potential nesting issues
        matches = list(re.finditer(pattern, content, re.IGNORECASE))
        for match in matches:
            start, end = match.span()
            # Check if match is already inside a link
            before = content[:start]
            after = content[end:]
            if '[[component:' in before[-20:] and ']]' in after[:20]:
                return True
        return False

class ComponentCoverageValidator:
    """Comprehensive validation for component coverage"""
    
    def __init__(self, walkthrough_state: WalkthroughState):
        self.walkthrough_state = walkthrough_state
        self.required_components = self._get_required_components()
    
    def _get_required_components(self) -> Dict[str, Dict]:
        """Get all components that should be linked"""
        required = {}
        
        for block_id, block_state in self.walkthrough_state.block_states.items():
            for comp_id, comp_state in block_state.component_states.items():
                comp_key = f"{block_id}:{comp_state.component_number}"
                required[comp_key] = {
                    "name": comp_state.component_name,
                    "type": comp_state.type,
                    "block_id": block_id,
                    "component_number": comp_state.component_number,
                    "description": comp_state.description
                }
        
        return required
    
    def validate_walkthrough(self, walkthrough: Walkthrough) -> Dict[str, any]:
        """Comprehensive validation of walkthrough"""
        
        validation_results = {
            "coverage_analysis": {},
            "link_quality": {},
            "missing_components": {},
            "broken_links": [],
            "duplicate_links": [],
            "overall_score": 0.0
        }
        
        # 1. Coverage Analysis
        linked_components = set()
        all_links = []
        
        for section in walkthrough.sections:
            section_links = self._extract_all_links(section.content)
            all_links.extend(section_links)
            
            for link in section_links:
                comp_key = f"{link['block_id']}:{link['component_number']}"
                linked_components.add(comp_key)
        
        # Calculate coverage
        total_required = len(self.required_components)
        total_linked = len(linked_components)
        coverage_percentage = (total_linked / total_required) * 100 if total_required > 0 else 0
        
        validation_results["coverage_analysis"] = {
            "total_components": total_required,
            "linked_components": total_linked,
            "coverage_percentage": coverage_percentage,
            "missing_count": total_required - total_linked
        }
        
        # 2. Find missing components
        missing_components = {}
        for comp_key, comp_info in self.required_components.items():
            if comp_key not in linked_components:
                block_id = comp_info["block_id"]
                if block_id not in missing_components:
                    missing_components[block_id] = []
                missing_components[block_id].append(comp_info)
        
        validation_results["missing_components"] = missing_components
        
        # 3. Link Quality Analysis
        validation_results["link_quality"] = self._analyze_link_quality(all_links)
        
        # 4. Find broken links (links to non-existent components)
        validation_results["broken_links"] = self._find_broken_links(all_links)
        
        # 5. Find duplicate links
        validation_results["duplicate_links"] = self._find_duplicate_links(all_links)
        
        # 6. Calculate overall score
        validation_results["overall_score"] = self._calculate_overall_score(validation_results)
        
        return validation_results
    
    def _extract_all_links(self, content: str) -> List[Dict]:
        """Extract all component links from content"""
        pattern = r'\[\[component:(\w+):(\d+):([^|]+)\|([^\]]+)\]\]'
        matches = re.findall(pattern, content)
        
        links = []
        for match in matches:
            links.append({
                "block_id": match[0],
                "component_number": int(match[1]),
                "component_name": match[2],
                "display_text": match[3],
                "full_link": f"[[component:{match[0]}:{match[1]}:{match[2]}|{match[3]}]]"
            })
        
        return links
    
    def _analyze_link_quality(self, links: List[Dict]) -> Dict:
        """Analyze quality of component links"""
        
        quality_metrics = {
            "total_links": len(links),
            "unique_components": len(set(f"{l['block_id']}:{l['component_number']}" for l in links)),
            "avg_display_text_length": 0,
            "descriptive_links": 0,
            "generic_links": 0
        }
        
        if links:
            total_length = sum(len(link["display_text"]) for link in links)
            quality_metrics["avg_display_text_length"] = total_length / len(links)
            
            for link in links:
                display_text = link["display_text"].lower()
                if any(word in display_text for word in ["function", "class", "method", "variable", "import"]):
                    quality_metrics["descriptive_links"] += 1
                else:
                    quality_metrics["generic_links"] += 1
        
        return quality_metrics
    
    def _find_broken_links(self, links: List[Dict]) -> List[Dict]:
        """Find links that point to non-existent components"""
        broken_links = []
        
        for link in links:
            comp_key = f"{link['block_id']}:{link['component_number']}"
            if comp_key not in self.required_components:
                broken_links.append({
                    "link": link,
                    "reason": "Component does not exist"
                })
        
        return broken_links
    
    def _find_duplicate_links(self, links: List[Dict]) -> List[Dict]:
        """Find duplicate links to the same component"""
        component_counts = defaultdict(list)
        
        for link in links:
            comp_key = f"{link['block_id']}:{link['component_number']}"
            component_counts[comp_key].append(link)
        
        duplicates = []
        for comp_key, link_list in component_counts.items():
            if len(link_list) > 1:
                duplicates.append({
                    "component_key": comp_key,
                    "count": len(link_list),
                    "links": link_list
                })
        
        return duplicates
    
    def _calculate_overall_score(self, validation_results: Dict) -> float:
        """Calculate overall validation score"""
        
        # Coverage score (0-40 points)
        coverage_score = min(40, validation_results["coverage_analysis"]["coverage_percentage"] * 0.4)
        
        # Quality score (0-30 points)
        quality = validation_results["link_quality"]
        if quality["total_links"] > 0:
            descriptive_ratio = quality["descriptive_links"] / quality["total_links"]
            quality_score = descriptive_ratio * 30
        else:
            quality_score = 0
        
        # Penalty for broken links (0-15 points deducted)
        broken_penalty = min(15, len(validation_results["broken_links"]) * 5)
        
        # Bonus for completeness (0-15 points)
        completeness_bonus = 15 if validation_results["coverage_analysis"]["coverage_percentage"] == 100 else 0
        
        total_score = max(0, coverage_score + quality_score - broken_penalty + completeness_bonus)
        return round(total_score, 1)

# Enhanced post-processing function that uses all the advanced features
def comprehensive_component_linking_postprocess(
    walkthrough: Walkthrough,
    walkthrough_state: WalkthroughState
) -> Tuple[Walkthrough, Dict]:
    """Apply comprehensive post-processing with all advanced features"""
    
    print("\n=== COMPREHENSIVE COMPONENT LINKING POST-PROCESSING ===")
    
    # 1. Build lookup dictionary
    print("Building component lookup dictionary...")
    lookup_dict = ComponentLookupDictionary(walkthrough_state)
    print(f"  ✓ Built dictionary with {len(lookup_dict.component_variations)} components")
    
    # 2. Initialize advanced linker
    print("Initializing advanced component linker...")
    linker = AdvancedComponentLinker(lookup_dict)
    print(f"  ✓ Created {len(linker.link_patterns)} linking patterns")
    
    # 3. Apply iterative replacement to each section
    total_replacements = 0
    for section in walkthrough.sections:
        print(f"Processing section: {section.block_name}")
        
        # Apply advanced linking
        new_content, replacements = linker.apply_iterative_replacement(section.content)
        section.content = new_content
        total_replacements += replacements
        
        print(f"  ✓ Made {replacements} component link improvements")
    
    print(f"  ✓ Total improvements: {total_replacements}")
    
    # 4. Validate results
    print("\nValidating component coverage...")
    validator = ComponentCoverageValidator(walkthrough_state)
    validation_results = validator.validate_walkthrough(walkthrough)
    
    # Update walkthrough metrics
    walkthrough.components_covered = validation_results["coverage_analysis"]["linked_components"]
    walkthrough.coverage_report = validation_results["missing_components"]
    
    print(f"  ✓ Coverage: {validation_results['coverage_analysis']['coverage_percentage']:.1f}%")
    print(f"  ✓ Overall score: {validation_results['overall_score']}/100")
    
    if validation_results["broken_links"]:
        print(f"  ⚠️  Found {len(validation_results['broken_links'])} broken links")
    
    if validation_results["duplicate_links"]:
        print(f"  ⚠️  Found {len(validation_results['duplicate_links'])} duplicate component links")
    
    print("\n=== POST-PROCESSING COMPLETE ===")
    
    return walkthrough, validation_results

# Legacy post-processing function (kept for compatibility)
def fix_component_links_in_walkthrough(walkthrough: Walkthrough) -> Walkthrough:
    """Legacy function - now redirects to comprehensive processing"""
    # This function is kept for backward compatibility
    # In practice, you should use comprehensive_component_linking_postprocess
    
    for section in walkthrough.sections:
        content = section.content
        block_id = section.block_id
        
        # Basic fixes for common issues
        if block_id == "3":
            content = re.sub(r'\b(?<!component:3:1:)VectorStore\b(?![|\]])', 
                           '[[component:3:1:VectorStore|VectorStore]]', content)
            content = re.sub(r'`append`\s+expression(?![|\]])', 
                           '[[component:3:5:append|append expression]]', content)
        
        elif block_id == "4":
            content = re.sub(r'`clean_text`\s+function(?![|\]])', 
                           '[[component:2:1:clean_text|clean_text function]]', content)
            content = re.sub(r'`tokenize`\s+function(?![|\]])', 
                           '[[component:2:2:tokenize|tokenize function]]', content)
        
        section.content = content
    
    return walkthrough

# Function to normalize component names for VS Code compatibility
def normalize_component_name_for_vscode(component_name: str) -> str:
    """Normalize component names to ensure VS Code extension compatibility"""
    normalized = component_name
    
    # For imports with spaces, convert to underscores
    if "Import" in normalized and " " in normalized:
        normalized = normalized.replace(" ", "_")
    
    return normalized

In [316]:
# 🔨 STEP 5: RUN WALKTHROUGH GENERATION AND SAVE COMPLETE RESULTS
print("\n📖 STEP 5: EDUCATIONAL WALKTHROUGH GENERATION")
print("="*60)

global walkthrough

# Ensure we have a complete state with all improvements
if 'improved_state' not in globals():
    print("⚠️  Running full analysis pipeline first...")
    # Run the complete pipeline if not already available
    graph = create_analysis_graph()
    initial_state = WalkthroughState(notebook_path=NOTEBOOK_TO_ANALYZE, code_overview="")
    result = await graph.ainvoke(initial_state)
    final_state = WalkthroughState(**result)
    
    # Generate deep descriptions
    for block_state in final_state.block_states.values():
        block_state.deep_description = None
    await generate_all_deep_descriptions(final_state)
    
    # Run iterative improvement
    improved_state = await smart_iterative_improvement(final_state, update_components=True)

print("🔧 Generating complete educational walkthrough...")

# Generate the enhanced walkthrough
walkthrough = await generate_complete_walkthrough(improved_state)

# Apply comprehensive post-processing
print("\n🔄 Applying comprehensive post-processing...")
final_walkthrough, validation_results = comprehensive_component_linking_postprocess(
    walkthrough, improved_state
)

# Save walkthrough results to ember_output
print("\n💾 Saving walkthrough results...")

# ✨ NEW: Save both original and preview versions using the new function
original_content, preview_content = save_walkthrough_with_preview(final_walkthrough, "complete_walkthrough")

# Save walkthrough data structure
walkthrough_data = {
    "introduction": final_walkthrough.introduction,
    "total_sections": len(final_walkthrough.sections),
    "total_components": final_walkthrough.total_components,
    "components_covered": final_walkthrough.components_covered,
    "coverage_percentage": (final_walkthrough.components_covered / final_walkthrough.total_components) * 100,
    "sections": []
}

# Save individual sections
for i, section in enumerate(final_walkthrough.sections):
    section_data = {
        "section_number": i + 1,
        "block_id": section.block_id,
        "block_name": section.block_name,
        "content": section.content,
        "components_referenced": section.components_referenced,
        "component_count": len(section.components_referenced)
    }
    
    walkthrough_data["sections"].append(section_data)
    
    # Save individual section files
    save_step("5_walkthrough", f"section_{i+1}_{section.block_id}.md", section.content, f"Section {i+1}: {section.block_name}")

save_step("5_walkthrough", "walkthrough_data.json", walkthrough_data, "Complete walkthrough data structure")

# Save validation and quality results
quality_report = {
    "validation_results": validation_results,
    "coverage_analysis": {
        "total_components": final_walkthrough.total_components,
        "components_covered": final_walkthrough.components_covered,
        "coverage_percentage": (final_walkthrough.components_covered / final_walkthrough.total_components) * 100,
        "missing_components": final_walkthrough.coverage_report
    },
    "link_quality": validation_results.get("link_quality", {}),
    "overall_score": validation_results.get("overall_score", 0)
}

save_step("5_walkthrough", "quality_report.json", quality_report, "Walkthrough quality and validation report")

# Create component link registry for VS Code extension
component_links = []
for section in final_walkthrough.sections:
    for comp_ref in section.components_referenced:
        component_links.append({
            "display_text": comp_ref["display_text"],
            "component_name": comp_ref["component_name"],
            "block_id": comp_ref["block_id"],
            "component_number": comp_ref["component_number"],
            "section_number": final_walkthrough.sections.index(section) + 1,
            "link_format": f"[[component:{comp_ref['block_id']}:{comp_ref['component_number']}:{comp_ref['component_name']}|{comp_ref['display_text']}]]"
        })

save_step("5_walkthrough", "component_links_registry.json", component_links, "Registry of all component links for VS Code extension")

# Display results summary
print(f"\n📊 WALKTHROUGH GENERATION RESULTS:")
print(f"   ✅ Sections generated: {len(final_walkthrough.sections)}")
print(f"   ✅ Components covered: {final_walkthrough.components_covered}/{final_walkthrough.total_components} ({(final_walkthrough.components_covered/final_walkthrough.total_components)*100:.1f}%)")
print(f"   ✅ Overall quality score: {validation_results.get('overall_score', 0)}/100")
print(f"   ✅ Total component links: {len(component_links)}")

# Save summary statistics
walkthrough_summary = {
    "notebook": NOTEBOOK_TO_ANALYZE,
    "sections_generated": len(final_walkthrough.sections),
    "total_components": final_walkthrough.total_components,
    "components_covered": final_walkthrough.components_covered,
    "coverage_percentage": (final_walkthrough.components_covered / final_walkthrough.total_components) * 100,
    "quality_score": validation_results.get("overall_score", 0),
    "total_links": len(component_links),
    "broken_links": len(validation_results.get("broken_links", [])),
    "duplicate_links": len(validation_results.get("duplicate_links", []))
}

save_step("5_walkthrough", "walkthrough_summary.json", walkthrough_summary, "Summary statistics for walkthrough generation")

# Show example section
if final_walkthrough.sections:
    first_section = final_walkthrough.sections[0]
    print(f"\n📄 EXAMPLE SECTION ({first_section.block_name}):")
    print("─" * 50)
    section_preview = first_section.content[:400] + "..." if len(first_section.content) > 400 else first_section.content
    print(section_preview)
    
    save_step("5_walkthrough", "section_preview.txt", section_preview, "Preview of first walkthrough section")

print(f"\n✅ Educational walkthrough generation complete!")
print(f"📁 All results saved to step5_walkthrough/ directory")
print(f"\n🎯 FOR CLEAN PREVIEW: Open 'complete_walkthrough_preview.md' in VS Code markdown preview")
print(f"   Components will appear as clickable blue links instead of raw syntax!")


📖 STEP 5: EDUCATIONAL WALKTHROUGH GENERATION
🔧 Generating complete educational walkthrough...

=== GENERATING ENHANCED WALKTHROUGH ===
  ✓ Built component name map with 189 components
Creating teaching plan...


  ✓ Created plan with 26 blocks

Generating block walkthroughs with mandatory component linking...
  ✓ Generated 25 block walkthroughs

=== ENHANCED WALKTHROUGH COMPLETE ===

🔄 Applying comprehensive post-processing...

=== COMPREHENSIVE COMPONENT LINKING POST-PROCESSING ===
Building component lookup dictionary...
  ✓ Built dictionary with 189 components
Initializing advanced component linker...
  ✓ Created 6324 linking patterns
Processing section: Environment Setup
  ✓ Made 3 component link improvements
Processing section: Document Loading
  ✓ Made 11 component link improvements
Processing section: Text Processing
  ✓ Made 10 component link improvements
Processing section: Embedding Model Initialization
  ✓ Made 13 component link improvements
Processing section: Vector Store Creation
  ✓ Made 12 component link improvements
Processing section: Retriever Setup
  ✓ Made 9 component link improvements
Processing section: Prompt Template Definition
  ✓ Made 13 component link improvements
Pr

In [317]:
# 🏁 FINALIZE EMBER ANALYSIS RUN
print("\n🏁 FINALIZING EMBER ANALYSIS")
print("="*60)

# Create final run summary and cleanup
finish_run()

print(f"\n🎉 EMBER ANALYSIS COMPLETE!")
print(f"📁 All results organized in ember_output directory")
print(f"🔍 Check ember_output/{NOTEBOOK_TO_ANALYZE.replace('.ipynb', '')}_* for timestamped results")
print(f"📊 Run summary and metadata saved")
print(f"🗑️  Old runs cleaned up (keeping 5 most recent)")

# Show final directory structure
print(f"\n📋 FINAL ANALYSIS STRUCTURE:")
print(f"   step1_extraction/    - Python code and markdown separated")
print(f"   step2_analysis/      - Block-level analysis and dependencies")
print(f"   step3_components/    - Component extraction and LangGraph results")
print(f"   step4_deep_descriptions/ - Enhanced descriptions and improvements") 
print(f"   step5_walkthrough/   - Educational walkthrough with component links")
print(f"   debug/               - Debug information and intermediate results")
print(f"   run_info.json        - Run metadata and progress tracking")
print(f"   run_summary.json     - Final summary with file inventory")

# List available runs for comparison
output_manager.list_runs()

print(f"\n✨ EMBER ANALYSIS PIPELINE COMPLETE ✨")
print(f"Ready for VS Code extension integration!")


🏁 FINALIZING EMBER ANALYSIS

📊 RUN SUMMARY
Directory: ember_output/multi_agent_20250806_072751
Total files: 105
  step4_deep_descriptions: 32 files
  step2_analysis: 6 files
  step5_walkthrough: 32 files
  step3_components: 31 files
  step1_extraction: 4 files
🗑️ Cleaning old run: tiny_demo_20250806_054044

🎉 EMBER ANALYSIS COMPLETE!
📁 All results organized in ember_output directory
🔍 Check ember_output/multi_agent_* for timestamped results
📊 Run summary and metadata saved
🗑️  Old runs cleaned up (keeping 5 most recent)

📋 FINAL ANALYSIS STRUCTURE:
   step1_extraction/    - Python code and markdown separated
   step2_analysis/      - Block-level analysis and dependencies
   step3_components/    - Component extraction and LangGraph results
   step4_deep_descriptions/ - Enhanced descriptions and improvements
   step5_walkthrough/   - Educational walkthrough with component links
   debug/               - Debug information and intermediate results
   run_info.json        - Run metadata an